# STEP 01. 개발환경 확인

## 작업 계획

이번 단계에서 확인할 것:

1. Python 버전, 플랫폼 정보 확인
2. `pandas`, `requests`, `beautifulsoup4` import 및 버전 확인

아직 크롤링 코드는 작성하지 않는다. 프로젝트 전용 가상환경(`chapter11/ax-job-agent/.venv`)이
제대로 구성되었는지만 확인한다.

In [1]:
# 1) Python 버전, 플랫폼 확인
import sys
import platform

print("Python version:", sys.version)
print("Platform:", platform.platform())
print("Executable:", sys.executable)

Python version: 3.14.7 (tags/v3.14.7:823f032, Aug  5 2026, 10:51:32) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
Executable: c:\dev\claude-code-agent-course\chapter11\ax-job-agent\.venv\Scripts\python.exe


## 패키지 import 및 버전 확인

STEP 04 이후 크롤링에서 사용할 핵심 패키지가 프로젝트 전용 가상환경에
정상적으로 설치되어 있는지 확인한다.

In [2]:
# 2) pandas, requests, beautifulsoup4 import 및 버전 확인
import pandas
import requests
import bs4

print("pandas version:", pandas.__version__)
print("requests version:", requests.__version__)
print("beautifulsoup4 version:", bs4.__version__)

pandas version: 3.0.6
requests version: 2.34.2
beautifulsoup4 version: 4.15.0


## 실행 결과 해석

- 성공 여부: 성공
- 확인한 데이터: Python 3.14.7 (Windows 11), 실행 파일 경로가 `chapter11/ax-job-agent/.venv`인 것을 확인 → 프로젝트 전용 가상환경이 맞음. pandas 3.0.6, requests 2.34.2, beautifulsoup4 4.15.0 모두 정상 import됨.
- 예상과 다른 부분: 없음. 계획대로 세 패키지 모두 에러 없이 import됨.
- 다음 단계 진행 가능 여부: 가능
- 추가 확인 사항: 없음


# STEP 03. 채용공고 페이지 접근 테스트

## 작업 계획

이번 단계에서 확인할 것:

1. 잡코리아 robots.txt를 requests로 가져와서, 검색 경로(/Search/)가 크롤링에 허용되는지 확인
2. 실제 검색 결과 페이지에 딱 1회만 요청을 보내서 status_code, Content-Type, 응답 길이를 확인
3. 응답 HTML 안에 채용공고로 보이는 텍스트가 실제로 들어있는지 일부 출력해서 확인

검색어는 "AX" 하나만 사용하고, 페이지도 1페이지만 확인한다. 여러 페이지를 반복 수집하는 것은 이번 단계의 범위가 아니다 (STEP 04에서 진행).

In [3]:
# 1) robots.txt 확인 - 검색 경로가 허용되는지 먼저 확인한다
import requests
from urllib.robotparser import RobotFileParser

SEARCH_KEYWORD = "AX"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    )
}

ROBOTS_URL = "https://www.jobkorea.co.kr/robots.txt"
SEARCH_URL = f"https://www.jobkorea.co.kr/Search/?stext={SEARCH_KEYWORD}"

robots_res = requests.get(ROBOTS_URL, headers=HEADERS, timeout=10)
print("robots.txt status_code:", robots_res.status_code)

rp = RobotFileParser()
rp.set_url(ROBOTS_URL)
rp.parse(robots_res.text.splitlines())

allowed = rp.can_fetch(HEADERS["User-Agent"], SEARCH_URL)
print("검색 대상 URL:", SEARCH_URL)
print("이 User-Agent로 크롤링 허용 여부:", allowed)

robots.txt status_code: 200
검색 대상 URL: https://www.jobkorea.co.kr/Search/?stext=AX
이 User-Agent로 크롤링 허용 여부: True


## 실제 검색 결과 페이지 요청 (1회만)

robots.txt에서 허용을 확인했으니, 같은 User-Agent로 실제 검색 결과 페이지에 딱 1번만 요청을 보낸다.
여러 페이지를 순회하거나 반복 요청하지 않는다.

In [4]:
# 2) 실제 검색 결과 페이지에 1회만 요청
response = requests.get(SEARCH_URL, headers=HEADERS, timeout=10)

print("status_code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("응답 길이 (len(response.text)):", len(response.text))

status_code: 200
Content-Type: text/html; charset=utf-8
응답 길이 (len(response.text)): 347872


In [5]:
# 3) 응답 HTML 안에 채용공고로 보이는 텍스트가 실제로 들어있는지 확인 (추가 요청 없이 같은 response 재사용)
from bs4 import BeautifulSoup

soup = BeautifulSoup(response.text, "html.parser")

print("페이지 제목:", soup.title.string if soup.title else None)

link_texts = [a.get_text(strip=True) for a in soup.find_all("a")]
job_like_texts = [t for t in link_texts if len(t) > 3][:10]

print("\n채용공고로 보이는 텍스트 샘플:")
for text in job_like_texts:
    print("-", text)

페이지 제목: 'AX' 관련 📢 채용공고 | 총 888건의 검색결과

채용공고로 보이는 텍스트 샘플:
- Skip to main content
- 회원가입/로그인
- [GS리테일] 9·10월 통합공고(BIZ CLUB팀 경력사원, MD AX 담당, 물류 AX 담당, 보안성 검토 담당)
- GS리테일GS그룹
- AX 컨설턴트 채용
- 에스코어삼성그룹
- 광고 프로덕트 기획 경력 채용(AX 광고 예산/과금 구조 설계 담당)
- ㈜NAVER네이버그룹
- [슈프리마HQ] HRD & AX 담당자 모집
- ㈜슈프리마


## 실행 결과 해석

- 성공 여부: 성공
- 확인한 데이터: robots.txt status_code 200, 검색 경로(`/Search/`)는 이 User-Agent(일반 브라우저)에 대해 `can_fetch = True`. 검색 페이지 status_code 200, Content-Type `text/html; charset=utf-8`, 응답 길이 348,347자. 페이지 제목에 "총 886건의 검색결과" 표시, GS리테일/에스코어/NAVER 등 실제 채용공고 텍스트 확인됨.
- 예상과 다른 부분: 없음. (robots.txt가 AI 크롤러 UA는 별도 차단한다는 사실은 `docs/PROJECT_SPEC.md` 9절에 기록)
- 다음 단계 진행 가능 여부: 가능
- 추가 확인 사항: 없음


# STEP 04. 소량 데이터 수집

## 작업 계획

STEP 03에서 검색어 "AX"로 이미 받은 검색 결과 페이지(`response`)를 그대로 재사용해서,
실제 공고 8개를 BeautifulSoup으로 파싱한다. 새로운 요청은 보내지 않는다.

추출할 항목:

- company_name (회사명)
- job_title (공고 제목)
- career (경력 조건)
- location (근무 지역)
- posted_date (등록일)
- job_url (공고 URL)

결과는 `list[dict]` 형태로 만들어 전체를 print로 출력한다.

In [6]:
# 1) STEP 03의 response를 재사용해서 공고 카드 파싱 (새 요청 없음)
soup = BeautifulSoup(response.text, "html.parser")

job_cards = soup.select("div.rounded-2xl.shadow-list.bg-white")[:8]
print("파싱 대상 공고 카드 수:", len(job_cards))


def extract_job(card):
    title_a = card.select_one('a[data-sentry-component="Title"]')
    job_title = title_a.get_text(strip=True) if title_a else None
    job_url = title_a["href"] if title_a else None

    company_span = card.select_one("span.mb-5 a span")
    company_name = company_span.get_text(strip=True) if company_span else None

    location_span = card.select_one('div[data-sentry-component="GrayChip"] span.truncate')
    location = location_span.get_text(strip=True) if location_span else None

    # 카드 상단에는 "믿고보는 대기업" 같은 배지가 같은 클래스를 쓰기도 해서,
    # 값이 2개 이상이면 실제 경력 조건은 두 번째 값이다.
    career_texts = [s.get_text(strip=True) for s in card.select("span.text-typo-c1-13")]
    career = career_texts[1] if len(career_texts) > 1 else (career_texts[0] if career_texts else None)

    # 검색 결과 목록 페이지(정적 HTML)에는 등록일 텍스트가 없다 (상세 페이지에서만 노출).
    # 추가 요청 없이 이 STEP을 완료해야 하므로 일단 None으로 둔다.
    posted_date = None

    return {
        "company_name": company_name,
        "job_title": job_title,
        "career": career,
        "location": location,
        "posted_date": posted_date,
        "job_url": job_url,
    }


jobs = [extract_job(card) for card in job_cards]

파싱 대상 공고 카드 수: 8


In [7]:
# 2) 파싱 결과 전부 출력 + company_name/job_title/job_url 값이 비어있지 않은지 확인
print(f"파싱된 공고 개수: {len(jobs)}\n")

for i, job in enumerate(jobs, start=1):
    print(f"[{i}] {job}")

missing = [
    job for job in jobs
    if not job["company_name"] or not job["job_title"] or not job["job_url"]
]
print("\ncompany_name/job_title/job_url이 비어있는 공고 수:", len(missing))

파싱된 공고 개수: 8

[1] {'company_name': 'GS리테일', 'job_title': '[GS리테일] 9·10월 통합공고(BIZ CLUB팀 경력사원, MD AX 담당, 물류 AX 담당, 보안성 검토 담당)', 'career': '경력3년↑', 'location': '서울 강남구 외 1', 'posted_date': None, 'job_url': 'https://www.jobkorea.co.kr/Recruit/GI_Read/50032017?Oem_Code=C1&logpath=1&stext=AX&listno=1&sc=630'}
[2] {'company_name': '에스코어', 'job_title': 'AX 컨설턴트 채용', 'career': '경력', 'location': '서울 송파구', 'posted_date': None, 'job_url': 'https://www.jobkorea.co.kr/Recruit/GI_Read/49589368?Oem_Code=C1&logpath=1&stext=AX&listno=2&sc=630'}
[3] {'company_name': '㈜NAVER', 'job_title': '광고 프로덕트 기획 경력 채용(AX 광고 예산/과금 구조 설계 담당)', 'career': '경력', 'location': '경기 성남시', 'posted_date': None, 'job_url': 'https://www.jobkorea.co.kr/Recruit/GI_Read/49976564?Oem_Code=C1&logpath=1&stext=AX&listno=3&sc=630'}
[4] {'company_name': '㈜슈프리마', 'job_title': '[슈프리마HQ] HRD & AX 담당자 모집', 'career': '경력7년↑', 'location': '경기 성남시', 'posted_date': None, 'job_url': 'https://www.jobkorea.co.kr/Recruit/GI_Read/49858859?Oem_Code=C1&

## 실행 결과 해석

- 성공 여부: 성공 (8개 공고 전부 파싱, 필수 항목 누락 0건)
- 확인한 데이터: `div.rounded-2xl.shadow-list.bg-white` 셀렉터로 공고 카드 8개 추출. company_name/job_title/job_url 8건 전부 정상. career, location도 대부분 채워짐. job_url에 `listno=1~8` 파라미터가 순서대로 붙어 있음.
- 예상과 다른 부분: 검색 결과 목록 페이지의 정적 HTML에는 등록일(posted_date) 텍스트가 없다 (아이콘/날짜 칩 자체가 존재하지 않음). 상세 페이지에서만 확인 가능할 것으로 보이며, 지금은 `None`으로 둔다.
- 다음 단계 진행 가능 여부: 가능 — job_url(신규 판별 기준)은 목록 페이지만으로 확보됨.
- 추가 확인 사항: posted_date를 채우려면 상세 페이지 요청이 추가로 필요함. 필요성이 실제로 확인되면(예: 기간 필터링) 별도 STEP으로 추가.


# STEP 04-1. 공고 카드 HTML 구조 재확인 (등록일/마감일 위치 찾기)

## 작업 계획

STEP 04에서 posted_date를 찾지 못해 `None`으로 두었다. 최종 파싱 코드를 고치기 전에, 카드 HTML 안에 날짜 텍스트가 실제로 있는지 눈으로 먼저 확인한다.

- 새 요청은 보내지 않는다. STEP 03의 `response`로 STEP 04에서 만든 `job_cards`를 그대로 쓴다.
- `job_cards[0]`(첫 번째 공고 카드) 하나만 `card.prettify()`로 전체 HTML을 출력한다.
- 사람이 눈으로 찾는 걸 돕기 위해 두 가지를 같이 출력한다.
  1. 카드 텍스트에서 날짜처럼 보이는 패턴("1일전", "~10/15", "D-3", "상시채용" 등)을 정규식으로 찾은 결과
  2. 카드 우측 하단 영역(경력/복리후생 줄 오른쪽에 있는 `div.flex-shrink-0`)의 HTML
- `extract_job()` 등 STEP 04의 파싱 코드는 이번 단계에서 고치지 않는다.

완료 조건: 첫 번째 카드 HTML에 등록일/마감일 텍스트가 있는지 없는지를 확실히 답할 수 있으면 된다.

In [8]:
# 1) 새 요청 없이 STEP 04의 job_cards[0] 하나만 전체 HTML 출력
import re

card = job_cards[0]

print("=" * 80)
print("job_cards[0].prettify()")
print("=" * 80)
print(card.prettify())

# 2) 눈으로 찾는 걸 돕기 위한 보조 출력: 날짜처럼 보이는 텍스트 패턴 검색
DATE_PATTERN = re.compile(
    r"\d+\s*(?:일|시간|분)\s*전"   # 1일전, 3시간 전
    r"|~\s*\d{1,2}/\d{1,2}"       # ~10/15
    r"|D-\d+"                     # D-3
    r"|오늘마감|내일마감|상시채용|채용시\s*마감"
    r"|\d{1,2}/\d{1,2}"           # 10/15
)
card_text = card.get_text(" ", strip=True)
print("=" * 80)
print("날짜 패턴 검색 결과:", DATE_PATTERN.findall(card_text))

# 3) 카드 우측 하단 영역(경력/복리후생 줄의 오른쪽 div) HTML
bottom_right = card.select("div.flex.flex-shrink-0")
print("카드 우측 하단 영역:", bottom_right[-1] if bottom_right else None)

job_cards[0].prettify()
<div class="w-full rounded-2xl p-0 shadow-list bg-white hover:bg-blue54" data-sentry-component="CardJob" data-sentry-source-file="index.tsx" style="cursor:pointer">
 <div class="flex flex-col">
  <div class="flex w-full gap-5 p-7">
   <a data-interactive="true" data-sentry-component="CompanyLogo" data-sentry-element="BaseLink" data-sentry-source-file="index.tsx" href="https://www.jobkorea.co.kr/Recruit/GI_Read/50032017?Oem_Code=C1&amp;logpath=1&amp;stext=AX&amp;listno=1&amp;sc=630" rel="noopener noreferrer" style="width:76px;height:76px" target="_blank">
    <div class="relative flex h-[76px] w-[76px] shrink-0 items-center justify-center overflow-hidden rounded-lg border border-gray200 bg-white px-1.5" style="border-radius:99px">
     <img alt="GS리테일 로고" data-nimg="1" data-sentry-component="Image" data-sentry-element="NextImage" data-sentry-source-file="index.tsx" decoding="async" height="0" loading="lazy" src="https://imgs.jobkorea.co.kr//Images/Logo/128/l/g/29

## 실행 결과 해석

> 아래 내용은 현재 열려 있는 노트북 커널에 이미 있던 `response`/`job_cards`로 같은 내용을 출력해서 확인한 결과다 (새 요청 없음). 커널이 이전 실행 때와 달라서 `len(response.text)`는 347,799로, 저장된 STEP 03 출력(348,347)과 조금 다르다. 첫 번째 카드는 이번에도 GS리테일 9·10월 통합공고였다.

- 성공 여부: 성공. `job_cards[0]` 전체 HTML을 출력했고, 날짜 텍스트가 있는지 없는지 판단할 수 있었다.
- 확인한 데이터/수치:
  - 카드 구조(위→아래): 로고 링크 → 배지("믿고보는 대기업") + 스크랩 버튼 → 제목(`a[data-sentry-component="Title"]`) → 회사명("GS리테일" / "GS그룹") → GrayChip 2개(근무지 "서울 강남구 외 1", 직무 "백화점·유통·도소매, CRM마케터, …") + "홈페이지 지원" 버튼 → 맨 아래 줄 왼쪽: "경력3년↑ • 휴양시설 지원, 의료비 지원, …"
  - **카드 우측 하단**(맨 아래 줄 오른쪽)은 `<div class="flex flex-shrink-0 gap-[2px]"></div>`로 **자식도 텍스트도 없는 빈 div**다.
  - 날짜 패턴 검색 결과: `[]`. "1일전", "~10/15", "D-3", "상시채용" 같은 텍스트는 카드 어디에도 없다.
- 예상과 다른 부분:
  - 등록일/마감일이 들어갈 것 같은 자리(우측 하단 div)는 있는데 서버가 준 HTML에서는 비어 있다. 브라우저에서 JavaScript가 나중에 채우는 영역으로 보인다 (추정, 아직 확인 안 함). 그래서 requests + BeautifulSoup으로는 목록 페이지에서 날짜를 얻을 수 없다.
  - STEP 04의 career 추출과 관련된 점: `span.text-typo-c1-13` 순서가 `['믿고보는 대기업', '경력3년↑', '•', '휴양시설 지원,…']`다. 배지가 있는 카드는 `[1]`이 경력이 맞지만, **배지가 없는 카드는 `[1]`이 "•"가 될 수 있다.** 경력 span만 `flex-shrink-0` 클래스를 갖고 있으므로 나중에 선택자를 바꿀 때 참고한다 (이번 단계에서는 고치지 않음).
- 다음 단계 진행 가능 여부: 가능. 목록 HTML에 posted_date가 없다는 게 확인되었으므로 STEP 04의 `None` 처리는 일단 유지하고 STEP 05로 넘어가도 된다.
- 추가 확인 사항:
  - 이번에 본 건 첫 번째 카드뿐이다. 나머지 7개 카드에도 우측 하단 div가 비어 있는지 확인이 필요하다.
  - 날짜가 필요하다면: (a) 공고 상세 페이지(`job_url`)를 추가로 요청해서 찾기, (b) 페이지 안의 `<script>` JSON(예: `__NEXT_DATA__`)에 날짜가 들어 있는지 찾아보기, (c) posted_date 컬럼 빼기 중에서 정해야 한다. (b)는 새 요청 없이 지금 `response`만으로 확인할 수 있다.
  - career 선택자를 배지 없는 카드에서도 맞게 동작하도록 바꿀지 검토한다.

# STEP 04-2. 전체 카드 HTML에서 날짜/기간 패턴 검색

## 작업 계획

STEP 04-1에서는 첫 번째 카드(`job_cards[0]`)의 `prettify()` 출력과 **텍스트**만 검사했다. 이번에는 범위를 넓혀 8개 카드 전부의 **HTML 원문**(태그 속성 포함)을 정규식으로 검색한다. 등록일/마감일이 화면 텍스트가 아니라 속성값에 숨어 있을 수도 있기 때문이다.

- 새 요청은 보내지 않는다. STEP 03의 `response`로 STEP 04에서 만든 `job_cards`를 그대로 쓴다.
- `job_cards[0]`의 전체 HTML 구조는 STEP 04-1 셀에서 이미 `prettify()`로 출력하므로 여기서 다시 출력하지 않는다.
- 검색 패턴: `\d+일\s*전`, `~\s*\d{1,2}/\d{1,2}`, `D-\d+`, `\d{2}\.\d{2}\.\d{2}`, 그리고 참고용으로 `\d{1,2}/\d{1,2}`, `\d{4}-\d{2}-\d{2}`
- 카드마다 우측 하단 영역(`div.flex-shrink-0`)이 비어 있는지도 같이 출력한다.
- STEP 04의 파싱 코드는 고치지 않는다.

완료 조건: 8개 카드 각각에 날짜/기간 패턴이 있는지 없는지와, 우측 하단 영역이 비어 있는지를 표로 확인할 수 있으면 된다.

In [9]:
# 1) 새 요청 없이 8개 카드 전체의 HTML 원문(str(card))에서 날짜/기간 패턴 검색
import re

DATE_PATTERNS = {
    "N일 전": r"\d+일\s*전",               # 1일전, 3일 전
    "~MM/DD": r"~\s*\d{1,2}/\d{1,2}",      # ~10/15
    "D-N": r"D-\d+",                       # D-3
    "YY.MM.DD": r"\d{2}\.\d{2}\.\d{2}",    # 25.10.15
    "MM/DD": r"\d{1,2}/\d{1,2}",           # 10/15 (참고용)
    "YYYY-MM-DD": r"\d{4}-\d{2}-\d{2}",    # 2025-10-15 (참고용)
}

print("검사 대상 카드 수:", len(job_cards))

for i, card in enumerate(job_cards):
    card_html = str(card)
    matches = {name: re.findall(p, card_html) for name, p in DATE_PATTERNS.items()}
    matches = {name: found for name, found in matches.items() if found}

    bottom_right = card.select("div.flex.flex-shrink-0")
    bottom_right_html = bottom_right[-1].decode_contents() if bottom_right else None

    print(f"\n[{i}] {card.select_one('a[data-sentry-component=\"Title\"]').get_text(strip=True)[:40]}")
    print("  날짜 패턴:", matches or "없음")
    print("  우측 하단 영역:", bottom_right_html or "(빈 div)")

검사 대상 카드 수: 8

[0] [GS리테일] 9·10월 통합공고(BIZ CLUB팀 경력사원, MD AX
  날짜 패턴: 없음
  우측 하단 영역: (빈 div)

[1] AX 컨설턴트 채용
  날짜 패턴: 없음
  우측 하단 영역: (빈 div)

[2] 광고 프로덕트 기획 경력 채용(AX 광고 예산/과금 구조 설계 담당)
  날짜 패턴: 없음
  우측 하단 영역: (빈 div)

[3] [슈프리마HQ] HRD & AX 담당자 모집
  날짜 패턴: 없음
  우측 하단 영역: (빈 div)

[4] 광고 프로덕트 기획 경력 채용(AX 광고 최적화 전략 담당)
  날짜 패턴: 없음
  우측 하단 영역: (빈 div)

[5] [캐럿글로벌] B2B AI·AX 교육사업 매니저
  날짜 패턴: 없음
  우측 하단 영역: (빈 div)

[6] [킨다그로스] AX 프로덕트 빌더
  날짜 패턴: 없음
  우측 하단 영역: (빈 div)

[7] [아시아경제] AX전략부 AX전략 및 데이터분석 담당자 채용
  날짜 패턴: 없음
  우측 하단 영역: (빈 div)


## 실행 결과 해석

> 아래 내용은 현재 열려 있는 노트북 커널에 이미 있던 `job_cards`(8개)로 같은 검사를 돌려 확인한 결과다 (새 요청 없음).

- 성공 여부: 성공. 8개 카드 전부를 검사했다.
- 확인한 데이터/수치:
  - 검사 대상 카드 수: 8
  - **8개 카드 모두 날짜 패턴 0건.** 화면 텍스트뿐 아니라 `href`, `src` 같은 태그 속성까지 포함한 HTML 원문 전체에서 `N일 전`, `~MM/DD`, `D-N`, `YY.MM.DD`, `MM/DD`, `YYYY-MM-DD` 중 아무것도 나오지 않았다.
  - **8개 카드 모두 우측 하단 `div.flex-shrink-0`이 빈 div**다. 첫 번째 카드만 그런 게 아니었다.
- 예상과 다른 부분:
  - 날짜가 속성값에 숨어 있을 가능성도 봤지만 없었다. 서버가 준 목록 HTML의 카드 안에는 등록일/마감일 정보가 아예 없는 것으로 판단한다.
  - 8개 카드 모두 같은 자리가 비어 있으므로, 브라우저에서 JavaScript가 채우는 영역이라는 추정에 힘이 실린다 (아직 확인은 안 함).
- 다음 단계 진행 가능 여부: 가능. 카드 HTML만으로는 posted_date를 채울 수 없다는 게 확정되었으므로, STEP 04의 `None` 처리를 유지한 채 STEP 05를 진행해도 된다.
- 추가 확인 사항:
  - 카드 밖, 페이지 전체(`response.text`)의 `<script>` JSON(예: `__NEXT_DATA__`)에 날짜가 들어 있는지는 아직 보지 않았다. 새 요청 없이 확인할 수 있는 마지막 후보다.
  - 그래도 없으면 공고 상세 페이지를 추가로 요청할지, posted_date 컬럼을 뺄지 정해야 한다.

# STEP 04b. posted_date, closing_date 보강 (Playwright)

## 작업 계획

STEP 04-1/04-2에서 `requests`로 받은 정적 HTML에는 등록일/마감일이 없다는 걸 확인했다. PROJECT_SPEC.md 10절 결정대로, Playwright로 JavaScript까지 실행된 화면의 HTML을 받아 **날짜 두 항목만** 보강한다.

1. Playwright sync API(headless=True)로 `SEARCH_URL`을 **1회만** 열고, 공고 카드와 날짜 텍스트("등록")가 나타날 때까지 기다린 뒤 `page.content()`로 렌더링된 HTML을 가져온다.
2. 렌더링된 HTML에서는 `posted_date`(등록일), `closing_date`(마감일) **두 항목만** 뽑는다.
   - 카드 우측 하단의 "09/11(금) 등록 • 09/29(화) 마감"을 "등록"/"마감" 텍스트 기준으로 나눈다.
   - "상시채용"처럼 날짜가 아닌 마감 표시는 텍스트 그대로 저장한다.
   - company_name/job_title/career/location/job_url은 다시 뽑지 않는다. 이 항목들은 STEP 04의 requests 기반 파싱이 정확했다. job_url은 merge 키로만 쓴다.
3. `job_url`을 키로 STEP 04의 원본 `jobs`(list[dict])에 posted_date, closing_date 두 컬럼만 추가한다. 결과 변수명은 **`jobs` 그대로** 유지한다 (별도 변수를 STEP 05로 넘기지 않음).
4. job_url 매칭이 안 되는 공고가 있으면 개수를 출력한다.

완료 조건: 8개 공고 전부 posted_date, closing_date가 채워진 `jobs` 리스트.

In [10]:
# 1) Playwright(sync API, headless)로 SEARCH_URL을 1회만 열어 JavaScript까지 실행된 HTML 가져오기
import asyncio
import sys
import warnings
from concurrent.futures import ThreadPoolExecutor

from playwright.sync_api import sync_playwright

CARD_SELECTOR = "div.rounded-2xl.shadow-list.bg-white"


def fetch_rendered_html(url):
    # Jupyter 커널 안에서는 sync API를 바로 못 쓰므로 별도 스레드에서 실행한다.
    # Windows 커널의 기본 이벤트 루프(Selector)는 브라우저 프로세스를 못 띄워서, 이 스레드에서만 Proactor로 바꿨다가 되돌린다.
    old_policy = None
    if sys.platform == "win32":
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", DeprecationWarning)
            old_policy = asyncio.get_event_loop_policy()
            asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    try:
        with sync_playwright() as p:
            browser = p.chromium.launch(headless=True)
            try:
                page = browser.new_page(user_agent=HEADERS["User-Agent"])
                page.goto(url, timeout=30_000)  # 요청은 이 1회뿐
                page.wait_for_selector(CARD_SELECTOR, timeout=15_000)  # 공고 카드 로드 대기
                page.wait_for_selector(f"{CARD_SELECTOR} >> text=/등록/", timeout=15_000)  # 날짜 렌더링 대기
                return page.content()
            finally:
                browser.close()
    finally:
        if old_policy is not None:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", DeprecationWarning)
                asyncio.set_event_loop_policy(old_policy)


with ThreadPoolExecutor(max_workers=1) as executor:
    rendered_html = executor.submit(fetch_rendered_html, SEARCH_URL).result()

print("요청 URL:", SEARCH_URL)
print("렌더링된 HTML 길이 (len(rendered_html)):", len(rendered_html))
print("참고 - STEP 03 정적 HTML 길이 (len(response.text)):", len(response.text))

요청 URL: https://www.jobkorea.co.kr/Search/?stext=AX
렌더링된 HTML 길이 (len(rendered_html)): 363171
참고 - STEP 03 정적 HTML 길이 (len(response.text)): 347872


## 렌더링된 HTML에서 날짜 두 항목만 추출 (새 요청 없음)

위에서 받은 `rendered_html`만 쓴다. 결과는 `job_url → {posted_date, closing_date}` 딕셔너리(`dates_by_url`)로 만든다.

In [11]:
# 2) 렌더링된 HTML에서 posted_date, closing_date 두 항목만 추출 (새 요청 없음)
rendered_soup = BeautifulSoup(rendered_html, "html.parser")
rendered_cards = rendered_soup.select(CARD_SELECTOR)[:8]
print("파싱 대상 공고 카드 수:", len(rendered_cards))


def extract_dates(card):
    # 카드 우측 하단 div: "09/11(금) 등록" / "•" / "09/29(화) 마감" (또는 "상시채용") span이 차례로 들어 있다
    date_area = card.select("div.flex.flex-shrink-0")
    texts = [s.get_text(strip=True) for s in date_area[-1].select("span")] if date_area else []

    posted_date = None
    closing_date = None
    for text in texts:
        if text.endswith("등록"):
            posted_date = text.removesuffix("등록").strip()
        elif text.endswith("마감"):
            closing_date = text.removesuffix("마감").strip()
        elif text != "•":
            closing_date = text  # "상시채용"처럼 날짜가 아닌 마감 표시는 텍스트 그대로
    return posted_date, closing_date


# job_url은 merge 키로만 쓴다 (나머지 항목은 STEP 04 값을 그대로 쓰므로 다시 뽑지 않음)
dates_by_url = {}
for card in rendered_cards:
    title_a = card.select_one('a[data-sentry-component="Title"]')
    if title_a is None:
        continue
    posted_date, closing_date = extract_dates(card)
    dates_by_url[title_a["href"]] = {"posted_date": posted_date, "closing_date": closing_date}

print("날짜를 추출한 job_url 수:", len(dates_by_url))

파싱 대상 공고 카드 수: 8
날짜를 추출한 job_url 수: 8


## STEP 04의 jobs에 merge (job_url 기준)

In [12]:
# 3) job_url을 키로 STEP 04의 jobs에 posted_date, closing_date 두 컬럼만 추가 (변수명 jobs 유지)
unmatched_urls = []
for job in jobs:
    dates = dates_by_url.get(job["job_url"])
    if dates is None:
        unmatched_urls.append(job["job_url"])
        job["posted_date"] = None
        job["closing_date"] = None
    else:
        job["posted_date"] = dates["posted_date"]
        job["closing_date"] = dates["closing_date"]

rendered_only_urls = dates_by_url.keys() - {job["job_url"] for job in jobs}
print("job_url 매칭 안 된 공고 수 (STEP 04 jobs 기준):", len(unmatched_urls))
for url in unmatched_urls:
    print("  -", url)
print("렌더링 쪽에만 있는 job_url 수:", len(rendered_only_urls))

print(f"\njobs 공고 수: {len(jobs)}\n")
for i, job in enumerate(jobs, start=1):
    print(f"[{i}] {job}")

missing_dates = [job for job in jobs if job["posted_date"] is None or job["closing_date"] is None]
print("\nposted_date/closing_date가 None인 공고 수:", len(missing_dates))

job_url 매칭 안 된 공고 수 (STEP 04 jobs 기준): 0
렌더링 쪽에만 있는 job_url 수: 0

jobs 공고 수: 8

[1] {'company_name': 'GS리테일', 'job_title': '[GS리테일] 9·10월 통합공고(BIZ CLUB팀 경력사원, MD AX 담당, 물류 AX 담당, 보안성 검토 담당)', 'career': '경력3년↑', 'location': '서울 강남구 외 1', 'posted_date': '09/21(월)', 'job_url': 'https://www.jobkorea.co.kr/Recruit/GI_Read/50032017?Oem_Code=C1&logpath=1&stext=AX&listno=1&sc=630', 'closing_date': '10/01(목)'}
[2] {'company_name': '에스코어', 'job_title': 'AX 컨설턴트 채용', 'career': '경력', 'location': '서울 송파구', 'posted_date': '07/15(수)', 'job_url': 'https://www.jobkorea.co.kr/Recruit/GI_Read/49589368?Oem_Code=C1&logpath=1&stext=AX&listno=2&sc=630', 'closing_date': '09/30(수)'}
[3] {'company_name': '㈜NAVER', 'job_title': '광고 프로덕트 기획 경력 채용(AX 광고 예산/과금 구조 설계 담당)', 'career': '경력', 'location': '경기 성남시', 'posted_date': '09/11(금)', 'job_url': 'https://www.jobkorea.co.kr/Recruit/GI_Read/49976564?Oem_Code=C1&logpath=1&stext=AX&listno=3&sc=630', 'closing_date': '09/29(화)'}
[4] {'company_name': '㈜슈프리마', 'job_title'

## 실행 결과 해석

> 아래 수치는 이전에 같은 Playwright 코드를 노트북 밖의 별도 스크립트로 1회 실행해 받은 렌더링 HTML(363,117자)과 STEP 04에 저장된 `jobs` 출력으로 확인한 결과다. 노트북에서 이 셀을 실행하면 요청이 1회 더 가고, 그 사이에 검색 결과가 바뀌면 수치가 달라질 수 있다. Windows Jupyter 커널에서 스레드 + Proactor 방식이 동작하는지는 `about:blank`로만 따로 확인했다 (사이트 요청 없음).

- 성공 여부: 성공. 완료 조건을 만족한다.
- 확인한 데이터/수치:
  - 파싱 대상 카드 8개. 날짜를 추출한 job_url 8개.
  - **job_url 매칭 안 된 공고 수: 0**, 렌더링 쪽에만 있는 job_url 수: 0
  - `jobs` 공고 수 8. **posted_date/closing_date가 None인 공고 0건.**
  - merge된 결과 (posted_date → closing_date):
    1. GS리테일: 09/21(월) → 10/01(목)
    2. 에스코어: 07/15(수) → 09/30(수)
    3. ㈜NAVER: 09/11(금) → 09/29(화)
    4. ㈜슈프리마: 09/02(수) → 10/25(일)
    5. ㈜NAVER: 09/11(금) → 09/29(화)
    6. ㈜캐럿글로벌: 08/14(금) → **상시채용** (날짜가 아니라서 텍스트 그대로 저장)
    7. ㈜킨다그로스: 09/07(월) → 10/31(토)
    8. ㈜아시아경제: 09/15(화) → 09/28(월)
  - company_name/job_title/career/location/job_url은 STEP 04 값 그대로다. 이번 셀은 이 값들을 건드리지 않는다.
- 예상과 다른 부분:
  - 없음.
  - 참고: 이전 버전(렌더링 HTML에서 전 항목을 다시 뽑는 방식)에서는, 날짜 span이 경력 span과 같은 클래스라서 배지 없는 카드 4개의 career가 "07/15(수) 등록"처럼 잘못 잡혔다. 날짜 두 항목만 뽑는 지금 방식에서는 이 문제가 생기지 않는다.
  - 날짜는 연도 없이 "MM/DD(요일)" 형태다.
  - 각 공고 dict의 키 순서: posted_date는 원래 자리(location 다음)에 있고, closing_date는 맨 뒤(job_url 다음)에 붙는다.
- 다음 단계 진행 가능 여부: 가능. STEP 05는 이제 날짜가 채워진 `jobs`를 그대로 받는다.
- 추가 확인 사항:
  - **merge 키인 job_url에는 검색 순번(`listno=N`)이 들어 있다.** requests 요청과 Playwright 요청 사이에 목록 순서가 바뀌면, 같은 공고라도 job_url이 달라서 매칭에 실패한다 (이번에는 0건). 매칭 실패가 생기면 `GI_Read/` 뒤의 공고번호를 키로 바꾸는 방법을 검토한다.
  - 이 셀은 `jobs`의 dict를 제자리에서 수정한다. STEP 04 셀을 다시 실행하면 날짜가 사라지므로 STEP 04b도 이어서 다시 실행해야 한다.
  - STEP 05의 결과 해석은 날짜를 보강하기 전 `jobs`로 쓴 것이다 (shape (8, 6), posted_date 결측 8). 노트북을 다시 실행하면 shape은 (8, 7), 날짜 결측은 0이 되어야 하므로 STEP 05 해석을 다시 써야 한다.
  - 날짜에 연도가 없으므로, 날짜 타입으로 바꾸려면 연도 규칙이 필요하다.
  - `asyncio` 이벤트 루프 정책 API는 Python 3.16에서 제거될 예정이라 그때 이 우회 코드도 바꿔야 한다.

# STEP 05. pandas DataFrame 변환

## 작업 계획

STEP 04b에서 posted_date, closing_date까지 채운 `jobs`(`list[dict]`)를 pandas DataFrame으로 변환하고, 구조를 점검한다. 새 요청은 보내지 않는다.

이번 단계에서 확인할 것:

1. `df.shape` — 행 수가 `len(jobs)`(8)와 같고, 열 수가 7(STEP 04 항목 6개 + STEP 04b의 closing_date)인지
2. `df.head()` — 한 행에 채용공고 한 건의 정보가 들어가 있는지
3. `df.dtypes` — 각 컬럼의 자료형
4. `df.isna().sum()` — 컬럼별 결측치 개수 (STEP 04b에서 날짜를 채웠으므로 전 컬럼 0이 예상됨)
5. `df.duplicated(subset=["job_url"]).sum()` — job_url 기준 중복 공고가 있는지

완료 조건: 행 수 = 공고 수, 컬럼 7개, 전 컬럼 결측 0, job_url 중복 0건.

In [13]:
# 1) STEP 04의 jobs(list[dict])를 DataFrame으로 변환하고 구조 점검
import pandas as pd

df = pd.DataFrame(jobs)

print("df.shape:", df.shape)
print("len(jobs):", len(jobs))

print("[df.head()]")
display(df.head())

print("[df.dtypes]")
print(df.dtypes)

print("[df.isna().sum()]")
print(df.isna().sum())

print("job_url 기준 중복 행 수:", df.duplicated(subset=["job_url"]).sum())

df.shape: (8, 7)
len(jobs): 8
[df.head()]


,company_name,job_title,career,location,posted_date,job_url,closing_date
0,GS리테일,"[GS리테일] 9·10월 통합공고(BIZ CLUB팀 경력사원, MD AX 담당, 물...",경력3년↑,서울 강남구 외 1,09/21(월),https://www.jobkorea.co.kr/Recruit/GI_Read/500...,10/01(목)
1,에스코어,AX 컨설턴트 채용,경력,서울 송파구,07/15(수),https://www.jobkorea.co.kr/Recruit/GI_Read/495...,09/30(수)
2,㈜NAVER,광고 프로덕트 기획 경력 채용(AX 광고 예산/과금 구조 설계 담당),경력,경기 성남시,09/11(금),https://www.jobkorea.co.kr/Recruit/GI_Read/499...,09/29(화)
3,㈜슈프리마,[슈프리마HQ] HRD & AX 담당자 모집,경력7년↑,경기 성남시,09/02(수),https://www.jobkorea.co.kr/Recruit/GI_Read/498...,10/25(일)
4,㈜NAVER,광고 프로덕트 기획 경력 채용(AX 광고 최적화 전략 담당),경력,경기 성남시,09/11(금),https://www.jobkorea.co.kr/Recruit/GI_Read/499...,09/29(화)


[df.dtypes]
company_name    str
job_title       str
career          str
location        str
posted_date     str
job_url         str
closing_date    str
dtype: object
[df.isna().sum()]
company_name    0
job_title       0
career          0
location        0
posted_date     0
job_url         0
closing_date    0
dtype: int64
job_url 기준 중복 행 수: 0


## 실행 결과 해석

> STEP 03 → 04 → 04b → 05를 노트북에서 차례로 실행한 결과다 (STEP 05 셀 실행 번호 13). STEP 04b는 job_url 매칭 실패 0건, 렌더링 쪽에만 있는 job_url 0건으로 `jobs` 8건 전부에 날짜를 채운 상태였다.

- 성공 여부: 성공. 완료 조건 4개를 모두 만족한다.
- 확인한 데이터/수치:
  - `df.shape` = `(8, 7)`, `len(jobs)` = 8 → 행 8개 = 공고 8건. 열 7개 = company_name, job_title, career, location, posted_date, job_url, closing_date
  - **한 행 = 채용공고 한 건** 재확인: `df.head()`의 0~4행이 STEP 04b 출력의 `[1]`~`[5]`와 1:1로 대응한다 (예: 0행 = GS리테일 9·10월 통합공고, 09/21(월) 등록 → 10/01(목) 마감). 행마다 job_url이 서로 다른 공고번호(`GI_Read/50032017`, `49589368`, …)를 가리키므로 URL로도 한 행이 공고 하나임이 확인된다.
  - `df.dtypes`: **7개 컬럼 전부 `str`.** 이전 실행에서 값이 전부 `None`이라 `object`였던 posted_date가 이제 `str`이다.
  - `df.isna().sum()`: **7개 컬럼 전부 0.** 이전 실행에서 8이었던 posted_date 결측이 0이 되었고, 새로 생긴 closing_date도 결측 0이다.
  - `df.duplicated(subset=["job_url"]).sum()` = 0 → job_url 기준 중복 없음
- 예상과 다른 부분:
  - 없음. 컬럼 수, 결측, 중복 모두 계획의 예상값과 같다.
  - 참고 1: 컬럼 순서가 `… posted_date, job_url, closing_date`로 closing_date가 맨 뒤에 있다. STEP 04b가 기존 dict 끝에 closing_date를 추가했기 때문이다. 동작에는 문제없다.
  - 참고 2: ㈜NAVER가 2행·4행 두 번 나오지만 job_title과 공고번호(49976564 / 49976547)가 다른 **별개 공고**라 중복이 아니다. 두 공고는 등록일/마감일(09/11(금) → 09/29(화))도 같다.
- 다음 단계 진행 가능 여부: 가능. 한 행 = 공고 한 건 구조이고, 7개 컬럼 모두 결측 없이 채워졌다.
- 추가 확인 사항:
  - posted_date/closing_date는 `"09/21(월)"`처럼 연도 없는 문자열이고, closing_date에는 `"상시채용"`(㈜캐럿글로벌) 같은 날짜가 아닌 값도 섞여 있다. 마감일로 정렬하거나 필터링하려면 날짜 타입 변환 규칙(연도 추정, "상시채용" 처리)을 정해야 한다.
  - job_url에 검색 순번(`listno=N`)이 붙어 있어서, 여러 페이지/여러 번 수집하면 같은 공고가 중복 검사를 빠져나갈 수 있다. 이후에는 공고번호 기준 중복 검사를 검토한다.
  - 컬럼 순서를 명세 순서로 맞출지(예: closing_date를 posted_date 옆으로) 정한다.

# STEP 06. 전처리 / 중복 제거

## 작업 계획

STEP 05의 `df`(8행 × 7열)를 전처리해서 `df_clean`을 만든다. 원본 `df`는 건드리지 않는다. 새 요청은 보내지 않는다.

1. **문자열 공백 제거**: company_name, job_title 등 모든 문자열 컬럼에 `str.strip()`을 적용한다.
2. **날짜 변환** (PROJECT_SPEC.md 10절 "연도 처리 규칙"): 원본 `posted_date`/`closing_date` 문자열은 그대로 두고, 연도를 붙인 datetime 컬럼 `posted_date_parsed`, `closing_date_parsed`를 추가한다.
   - posted_date: 올해 연도로 계산한 날짜가 오늘보다 미래면 연도 -1
   - closing_date: **posted_date_parsed와 같은 연도**로 계산한 날짜가 posted_date_parsed보다 이르면 연도 +1 (기준은 '오늘'이 아니라 등록일의 연도. 예: 2027-01-05에 "12/01 등록 • 12/31 마감"을 보면 2026-12-01 → 2026-12-31)
   - "상시채용" 등 날짜 형식이 아닌 값은 `NaT`로 두고, 원문은 `closing_date`에 남는다.
   - 알려진 한계: "02/29"가 윤년이 아닌 연도로 계산되면 에러가 난다. 현재는 예외 처리하지 않는다 (PROJECT_SPEC.md 10절).
3. **중복 제거**: `job_url` 기준 `drop_duplicates`를 적용한다. 이번 데이터엔 중복이 없지만 코드는 항상 포함한다.
4. 전처리 전후 행 수를 출력해서 비교한다.
5. 검증용 출력: 괄호 속 요일(예: "(월)")과 변환된 날짜의 실제 요일이 맞는지 대조한다. 연도가 틀리면 요일도 어긋나므로 연도 계산 검증이 된다 (출력만 하고 컬럼으로 저장하지 않음).

완료 조건: 전처리 후에도 8행 유지, `posted_date_parsed`에 실제 연도가 들어간 값을 확인할 수 있을 것.

In [ ]:
# 1) STEP 05의 df 전처리: 문자열 공백 제거 → 날짜 연도 부여 → job_url 기준 중복 제거
import re

rows_before = len(df)
df_clean = df.copy()

# (1) 모든 문자열 컬럼 앞뒤 공백 제거
str_cols = [col for col in df_clean.columns if pd.api.types.is_string_dtype(df_clean[col])]
for col in str_cols:
    df_clean[col] = df_clean[col].str.strip()
print("공백 제거한 문자열 컬럼:", str_cols)

# (2) 연도 처리 규칙 (PROJECT_SPEC.md 10절) - 원본 posted_date/closing_date 문자열은 그대로 두고 새 컬럼에 저장
MMDD_PATTERN = re.compile(r"^(\d{1,2})/(\d{1,2})")
today = pd.Timestamp.today().normalize()


def parse_mmdd(text, year):
    # 알려진 한계: "02/29"가 윤년이 아닌 year로 들어오면 ValueError (PROJECT_SPEC.md 10절)
    match = MMDD_PATTERN.match(text) if isinstance(text, str) else None
    if match is None:
        return pd.NaT  # "상시채용" 등 날짜 형식이 아닌 값
    return pd.Timestamp(year=year, month=int(match[1]), day=int(match[2]))


def parse_posted_date(text):
    posted = parse_mmdd(text, today.year)
    if posted is not pd.NaT and posted > today:  # 등록일은 미래일 수 없다 → 연도 -1
        posted = posted.replace(year=posted.year - 1)
    return posted


def parse_closing_date(text, posted):
    # 기준 연도는 '오늘'이 아니라 등록일의 연도 (등록일이 없으면 올해)
    base_year = posted.year if posted is not pd.NaT else today.year
    closing = parse_mmdd(text, base_year)
    if closing is not pd.NaT and posted is not pd.NaT and closing < posted:  # 마감일은 등록일보다 이를 수 없다 → 연도 +1
        closing = closing.replace(year=closing.year + 1)
    return closing


df_clean["posted_date_parsed"] = df_clean["posted_date"].map(parse_posted_date)
df_clean["closing_date_parsed"] = [
    parse_closing_date(text, posted)
    for text, posted in zip(df_clean["closing_date"], df_clean["posted_date_parsed"])
]
df_clean["posted_date_parsed"] = pd.to_datetime(df_clean["posted_date_parsed"])
df_clean["closing_date_parsed"] = pd.to_datetime(df_clean["closing_date_parsed"])

# (3) job_url 기준 중복 제거 (이번 데이터엔 없지만 항상 실행)
df_clean = df_clean.drop_duplicates(subset=["job_url"], keep="first").reset_index(drop=True)

print("\n기준일(today):", today.date())
print("전처리 전 행 수:", rows_before)
print("전처리 후 행 수:", len(df_clean))
print("제거된 행 수:", rows_before - len(df_clean))

## 날짜 변환 결과 확인

원본 문자열과 변환된 날짜를 나란히 놓고, 연도와 요일이 맞는지 확인한다.

In [ ]:
# 2) 날짜 변환 결과 확인 - 원본 문자열과 변환 결과를 나란히 보고, 괄호 속 요일과 변환된 날짜의 요일이 맞는지 대조
WEEKDAYS = "월화수목금토일"


def weekday_matches(text, parsed):
    if parsed is pd.NaT or not isinstance(text, str):
        return None  # 비교 대상 아님
    return f"({WEEKDAYS[parsed.weekday()]})" in text


date_check = df_clean[["company_name", "posted_date", "posted_date_parsed", "closing_date", "closing_date_parsed"]].copy()
date_check["posted_요일일치"] = [weekday_matches(t, p) for t, p in zip(df_clean["posted_date"], df_clean["posted_date_parsed"])]
date_check["closing_요일일치"] = [weekday_matches(t, p) for t, p in zip(df_clean["closing_date"], df_clean["closing_date_parsed"])]
display(date_check)

print("[df_clean.dtypes]")
print(df_clean.dtypes)
print("\n[df_clean.isna().sum()]")
print(df_clean.isna().sum())
print("\n[posted_date_parsed 연도 분포]")
print(df_clean["posted_date_parsed"].dt.year.astype("Int64").value_counts(dropna=False).to_string())
print("[closing_date_parsed 연도 분포]")
print(df_clean["closing_date_parsed"].dt.year.astype("Int64").value_counts(dropna=False).to_string())
print("마감일 < 등록일인 행 수:", (df_clean["closing_date_parsed"] < df_clean["posted_date_parsed"]).sum())

## 실행 결과 해석

> 아래 수치는 노트북에 저장된 STEP 04b 출력(날짜가 병합된 `jobs` 8건)으로 `df`를 다시 만들어, 같은 코드를 노트북 밖에서 돌려 확인한 결과다 (기준일 2026-09-23). 노트북에서 직접 실행하면 기준일(`today`)이 실행한 날짜로 바뀐다.

- 성공 여부: 성공. 완료 조건을 만족한다.
- 확인한 데이터/수치:
  - **전처리 전 행 수 8 → 전처리 후 행 수 8**, 제거된 행 0. job_url 중복이 없어서 `drop_duplicates`로 빠진 행이 없다.
  - 공백 제거 대상 문자열 컬럼 7개: company_name, job_title, career, location, posted_date, job_url, closing_date
  - **posted_date_parsed에 연도가 들어갔다**: 8건 전부 2026년이다 (예: "09/21(월)" → 2026-09-21, "07/15(수)" → 2026-07-15). 기준일(2026-09-23)보다 미래인 등록일이 없어서 연도 -1이 적용된 행은 없다.
  - closing_date_parsed: 7건 2026년, 1건 NaT (㈜캐럿글로벌 "상시채용"). 원문 "상시채용"은 `closing_date`에 그대로 남아 있다. 마감일이 등록일보다 이른 행이 없어서 연도 +1이 적용된 행도 없다.
  - **요일 대조**: 변환된 날짜 15개(등록일 8 + 마감일 7)의 실제 요일이 괄호 속 요일과 전부 일치한다. 2026년이라는 연도 계산이 맞다는 뜻이다.
  - dtypes: 원본 7개 컬럼은 `str`, 새 컬럼 2개는 `datetime64[us]`
  - 결측: closing_date_parsed만 1 (상시채용), 나머지 0
  - 마감일 < 등록일인 행: 0
- 예상과 다른 부분:
  - 없음.
  - 이번 데이터는 원래 앞뒤 공백이 없어서 strip으로 바뀐 값은 없다. 코드는 테스트용으로 company_name에 공백을 넣어 제거되는 것을 따로 확인했다.
  - 중복 제거도, 테스트용으로 행 하나를 복제한 9행 데이터에서 8행으로 줄어드는 것을 따로 확인했다.
- 다음 단계 진행 가능 여부: 가능. 이후 STEP에서는 `df_clean`을 쓴다.
- 추가 확인 사항:
  - 연말 경계 케이스는 기준일을 2027-01-05로 바꿔 따로 확인했다. "12/20 등록 • 01/10 마감"은 2026-12-20 → 2027-01-10, "12/01 등록 • 12/31 마감"은 2026-12-01 → 2026-12-31로 계산된다. closing_date의 기준 연도를 posted_date_parsed의 연도로 잡는 방식은 PROJECT_SPEC.md 10절 규칙 문구에도 반영했다.
  - 기준일이 "실행한 날"이라서, 오래된 데이터를 나중에 다시 처리하면 연도가 달라질 수 있다. PROJECT_SPEC.md 3절의 `collected_at`(수집 시각)을 기준일로 쓰는 게 맞다. 다만 `search_keyword`, `collected_at` 두 컬럼은 명세에는 있지만 아직 수집 코드에 없다.
  - 알려진 한계: "02/29"가 윤년이 아닌 해로 계산되면 에러가 난다. 드문 경우라 현재는 예외 처리하지 않는다 (PROJECT_SPEC.md 10절에 기록).

# STEP 07. 신규 공고 판별 (jobs_history.csv 기준)

## 작업 계획

STEP 06의 `df_clean`을 `data/processed/jobs_history.csv`와 비교해서 이번에 처음 본 공고를 표시한다. 새 요청은 보내지 않는다.

1. **job_id 추출**: `job_url`에서 `GI_Read/` 뒤의 숫자(공고 ID)만 뽑아 `job_id` 컬럼을 만든다.
   - `job_url` 전체를 키로 쓰지 않는 이유: `listno=N` 같은 검색 순번 파라미터는 실행할 때마다 바뀔 수 있다. 같은 공고라도 job_url 문자열이 달라질 수 있으므로, 안정적인 키는 공고 ID다.
2. **history 비교**:
   - 파일이 없으면 새로 만들고, `df_clean` 전체를 신규(`is_new=True`)로 표시해 저장한다.
   - 파일이 있으면 기존 history의 `job_id`와 비교해 `is_new`(True/False)를 추가하고 신규 건수를 출력한다.
   - job_id를 못 뽑은(NaN) 행은 비교할 수 없으므로 `is_new=False`로 두고 history에도 넣지 않는다. 몇 건인지는 출력으로 확인한다.
3. **history 갱신**: 신규 행만 기존 history 뒤에 append해서 다시 저장한다. 같은 job_id가 이미 있으면 기존 행을 덮어쓰지 않고 유지한다.
4. 저장한 파일을 다시 읽어서 행 수, job_id 중복/결측을 확인한다.

- 경로: 노트북 커널의 작업 폴더가 `notebooks/`이므로, 프로젝트 루트(`chapter11/ax-job-agent/`) 기준 `data/processed/jobs_history.csv`로 잡는다. 폴더가 없으면 만든다.
- 인코딩: 엑셀에서 한글이 깨지지 않도록 `utf-8-sig`로 저장한다.

완료 조건: 전체 건수, 신규 건수, job_id 추출 실패(NaN) 건수를 실제 숫자로 확인하고, jobs_history.csv가 job_id 중복 없이 저장될 것.

In [ ]:
# 1) job_url에서 공고 ID(GI_Read/ 뒤 숫자)를 뽑아 job_id 컬럼 추가
from pathlib import Path

df_clean["job_id"] = df_clean["job_url"].str.extract(r"GI_Read/(\d+)", expand=False)

total_count = len(df_clean)
job_id_missing = df_clean["job_id"].isna().sum()
print("전체 건수:", total_count)
print("job_id 추출 실패(NaN) 건수:", job_id_missing)
print("job_id 고유값 수:", df_clean["job_id"].nunique())
display(df_clean[["company_name", "job_id", "job_url"]])

## history와 비교 → is_new 표시 → 저장

주의: 이 셀은 `jobs_history.csv`를 쓴다. 첫 실행에서는 전부 신규가 되고, 두 번째 실행부터는 같은 공고가 `is_new=False`가 된다.

In [ ]:
# 2) jobs_history.csv와 job_id를 비교해 is_new 표시 → 신규 공고만 history에 append해서 저장
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
HISTORY_PATH = PROJECT_ROOT / "data" / "processed" / "jobs_history.csv"

history_existed = HISTORY_PATH.exists()
if history_existed:
    history = pd.read_csv(
        HISTORY_PATH,
        dtype={"job_id": str},
        parse_dates=["posted_date_parsed", "closing_date_parsed"],  # 날짜 형식을 새 행과 맞춘다
    )
    known_ids = set(history["job_id"].dropna())
    print("기존 history 파일:", HISTORY_PATH)
    print("기존 history 행 수:", len(history))
else:
    history = pd.DataFrame(columns=[*df_clean.columns, "is_new"])
    known_ids = set()
    print("history 파일 없음 → 새로 생성:", HISTORY_PATH)

# job_id가 NaN인 행은 비교할 수 없으므로 신규로 치지 않고 history에도 넣지 않는다
df_clean["is_new"] = df_clean["job_id"].notna() & ~df_clean["job_id"].isin(known_ids)
new_count = int(df_clean["is_new"].sum())
print("\n이번 수집 건수:", len(df_clean))
print("신규 건수 (is_new=True):", new_count)
print("기존 공고 건수 (is_new=False, job_id 있음):", int((~df_clean["is_new"] & df_clean["job_id"].notna()).sum()))

# 기존 행은 그대로 두고 신규 행만 뒤에 붙인다 (같은 job_id가 있으면 기존 행 유지)
new_rows = df_clean[df_clean["is_new"]]
history_updated = pd.concat([history, new_rows], ignore_index=True) if len(history) else new_rows.reset_index(drop=True)
history_updated = history_updated.drop_duplicates(subset=["job_id"], keep="first")

HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
history_updated.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")
print("\nhistory 저장 행 수:", len(history) if history_existed else 0, "→", len(history_updated))

## 저장된 history 파일 확인

In [ ]:
# 3) 저장된 jobs_history.csv를 다시 읽어서 확인
saved = pd.read_csv(HISTORY_PATH, dtype={"job_id": str})
print("저장된 파일:", HISTORY_PATH)
print("saved.shape:", saved.shape)
print("컬럼:", list(saved.columns))
print("job_id 중복 수:", saved["job_id"].duplicated().sum())
print("job_id 결측 수:", saved["job_id"].isna().sum())
display(saved[["job_id", "company_name", "posted_date_parsed", "closing_date_parsed", "is_new"]])

## 실행 결과 해석

> 아래 숫자는 노트북 커널에서 위 셀들을 실행한 결과다 (첫 실행). 저장된 `data/processed/jobs_history.csv`를 따로 읽어서 같은 숫자인지도 확인했다 (8행 × 11열, job_id 중복 0, 결측 0, is_new 8건 전부 True).

- 성공 여부: 성공. 완료 조건을 만족한다.
- 확인한 데이터/수치:
  - **전체 건수: 8**
  - **신규 건수: 8** — history 파일이 없던 첫 실행이라 8건 전부 `is_new=True`
  - **job_id 추출 실패(NaN) 건수: 0** — 8건 모두 `GI_Read/` 뒤 공고 ID를 뽑았다 (예: GS리테일 → 50032017)
  - history 파일: 새로 생성, 0행 → 8행 (11개 컬럼 = df_clean 9개 + job_id + is_new)
  - 저장된 파일 재확인: job_id 중복 0건, 결측 0건
- 예상과 다른 부분:
  - 없음. 첫 실행이라 전부 신규인 것은 예상한 결과다.
- 다음 단계 진행 가능 여부: 가능. STEP 08부터는 `is_new=True`인 공고만 분석한다.
- 추가 확인 사항:
  - 이제 history 파일이 있으므로, 이 STEP을 다시 실행하면 같은 공고 8건은 `is_new=False`(신규 0건)가 된다. 첫 실행 결과를 다시 보려면 `data/processed/jobs_history.csv`를 지우고 실행한다.
  - history의 `is_new`는 "추가될 때 신규였다"는 뜻이라 항상 True다. 나중에 언제 처음 봤는지가 필요하면 `first_seen_at` 같은 컬럼으로 바꾸는 것을 검토한다.

# STEP 08. 기본 분석 / 관련 공고 필터링

## 작업 계획

이번 STEP부터는 `df_clean` 전체가 아니라 **신규 공고(`is_new=True`)만** 분석한다 (STEP 07 결과 사용). 모든 수치는 pandas로 계산하고 Gemini는 쓰지 않는다 (PROJECT_SPEC.md 6절 원칙 7). 새 요청은 보내지 않고, 파일도 쓰지 않는다.

1. 신규 공고만 걸러낸 `df_new`를 만들고 신규 공고 건수를 출력한다.
2. `career`(경력 조건) 값별 건수 (`value_counts`)
3. `location`(근무 지역) 값별 건수
4. 마감 임박 순 정렬: `closing_date_parsed`가 빠른 순으로 상위 5건 (company_name, job_title, closing_date_parsed). 참고용으로 기준일(STEP 06의 `today`)부터 남은 일수 `days_left`도 같이 출력한다.
5. 마감일이 없는(NaT, 예: 상시채용) 공고 건수

- 주의: `jobs_history.csv`가 이미 있는 상태에서 STEP 07을 다시 실행하면 같은 공고가 전부 `is_new=False`가 되어, 이 STEP의 통계가 0건이 된다. 그럴 때는 경고 문구가 출력된다.

완료 조건: 위 5가지 통계를 실제 숫자로 확인할 것.

In [ ]:
# 1) 분석 대상: STEP 07에서 is_new=True로 표시된 신규 공고만
df_new = df_clean[df_clean["is_new"]].copy()

print("전체 공고 건수 (df_clean):", len(df_clean))
print("신규 공고 건수 (df_new):", len(df_new))
if df_new.empty:
    print("\n⚠️ 신규 공고가 0건이라 아래 통계가 전부 비어 있다.")
    print("   jobs_history.csv가 이미 있는 상태에서 STEP 07을 다시 실행하면 같은 공고는 신규가 아니게 된다.")

## 경력 조건 / 근무 지역 분포

In [ ]:
# 2) career(경력 조건), location(근무 지역) 값별 건수
print("[career 값별 건수]")
print(df_new["career"].value_counts(dropna=False).to_string())

print("\n[location 값별 건수]")
print(df_new["location"].value_counts(dropna=False).to_string())

## 마감 임박 공고 / 마감일 없는 공고

In [ ]:
# 3) 마감 임박 순 정렬 (closing_date_parsed가 빠른 순 상위 5건) + 마감일 없는(NaT) 공고 건수
TOP_N = 5

has_deadline = df_new.dropna(subset=["closing_date_parsed"])
closing_soon = has_deadline.sort_values("closing_date_parsed").head(TOP_N).copy()
closing_soon["days_left"] = (closing_soon["closing_date_parsed"] - today).dt.days  # today: STEP 06의 기준일

print(f"[마감 임박 상위 {TOP_N}건] (기준일: {today.date()})")
display(closing_soon[["company_name", "job_title", "closing_date_parsed", "days_left"]])

no_deadline = df_new[df_new["closing_date_parsed"].isna()]
print("마감일이 없는(NaT) 공고 건수:", len(no_deadline))
print("  원문 closing_date 값:", no_deadline["closing_date"].tolist())

## 실행 결과 해석

> 아래 숫자는 노트북 커널에서 위 셀들을 실행한 결과다 (STEP 07 첫 실행 직후의 `df_clean` 기준, 기준일 2026-09-23).

- 성공 여부: 성공. 계획한 5가지 통계를 모두 pandas로 계산했다 (Gemini 사용 없음).
- 확인한 데이터/수치:
  - **신규 공고 건수: 8** (전체 8건 중 8건. STEP 07 첫 실행이라 전부 신규)
  - **career 분포**: 경력 3 / 경력3년↑ 2 / 경력7년↑ 1 / 경력무관 1 / 신입·경력1년↑ 1 → 합계 8. "경력" 계열이 대부분이고, 신입이 지원 가능한 공고는 "경력무관"과 "신입·경력1년↑" 2건이다.
  - **location 분포**: 경기 성남시 3 / 서울 강남구 외 1, 서울 송파구, 서울 용산구, 서울 마포구, 서울 중구 각 1 → 합계 8. 성남시 3건 중 2건은 같은 회사(㈜NAVER) 공고다.
  - **마감 임박 상위 5건**:
    1. ㈜아시아경제 — 2026-09-28 (5일 남음)
    2. ㈜NAVER (AX 광고 예산/과금 구조 설계 담당) — 2026-09-29 (6일)
    3. ㈜NAVER (AX 광고 최적화 전략 담당) — 2026-09-29 (6일)
    4. 에스코어 — 2026-09-30 (7일)
    5. GS리테일 — 2026-10-01 (8일)
  - **마감일 없는(NaT) 공고: 1건** (㈜캐럿글로벌, 원문 "상시채용")
- 예상과 다른 부분:
  - 없음. 마감일 없는 1건은 STEP 06에서 확인한 상시채용 공고와 같다.
- 다음 단계 진행 가능 여부: **가능.**
- 추가 확인 사항:
  - `days_left`는 노트북을 실행한 날(STEP 06의 `today`) 기준이라, 다른 날 실행하면 값이 달라진다.
  - 이 통계는 신규 공고 기준이라, STEP 07을 다시 실행하면(신규 0건) 전부 비게 된다.
  - 이번 STEP의 "관련 공고 필터링"은 신규 공고만 걸러내는 것까지만 했다. 키워드 기준 관련성 필터가 필요하면 기준을 정해야 한다.

# STEP 09. Gemini API 연동

## 작업 계획

Gemini API를 처음 연결하고, 신규 공고 **1건만** 보내 한 줄 요약을 받아오는지 테스트한다. 전체 8건을 한번에 보내지 않는다.

노트북 밖에서 먼저 준비한 것 (완료):
- 프로젝트 폴더 `.gitignore` 생성: `.venv/`, `.env`, `__pycache__/`, `*.pyc`, `.ipynb_checkpoints/`. `data/`는 넣지 않는다. `jobs_history.csv`는 파이프라인 상태 파일이라 git으로 추적한다 (주 1회 자동 실행 때 이전 기록이 필요함).
- `chapter11/.gitignore` 생성: 상위 폴더에 있던 `chapter11/.env`(GEMINI_API_KEY 들어 있음)도 커밋되지 않게 한다.
- `.env.example` 생성: 실제 키 없이 `GEMINI_API_KEY=`만 적었다.
- SDK 설치: `google-genai` 2.25.0 (Google의 최신 Gen AI Python SDK, `from google import genai`)

이번 셀들에서 할 것:
1. python-dotenv로 `.env`를 읽어 `GEMINI_API_KEY`를 환경변수로 로드한다. 프로젝트 폴더의 `.env`를 먼저 찾고, 없으면 `chapter11/.env`를 쓴다. **키 값은 출력하지 않고, 로드 성공 여부만 True/False로 출력한다.**
2. `df_new`의 첫 번째 공고 1건의 company_name + job_title + career를 모델 `gemini-3.5-flash-lite`에 보내 한 줄 요약을 받는다. 정보에 없는 내용은 추측하지 말라고 프롬프트에 적는다.

절대 하지 않을 것: API 키를 코드에 하드코딩하기, print하기, git에 커밋되는 파일에 남기기.

완료 조건: API 키 로드 성공 True, 공고 1건 원문과 Gemini 한 줄 요약을 실제로 확인할 것.

In [ ]:
# 1) python-dotenv로 .env를 읽어 GEMINI_API_KEY 환경변수 로드 (키 값은 절대 출력하지 않는다)
import os
from pathlib import Path

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# 프로젝트 폴더(.env.example 옆)를 먼저 찾고, 없으면 상위 chapter11 폴더의 .env를 쓴다
ENV_CANDIDATES = [PROJECT_ROOT / ".env", PROJECT_ROOT.parent / ".env"]

env_path = next((path for path in ENV_CANDIDATES if path.exists()), None)
if env_path is not None:
    load_dotenv(env_path)

print(".env 파일 위치:", env_path.relative_to(PROJECT_ROOT.parent.parent).as_posix() if env_path else "찾지 못함")
print("GEMINI_API_KEY 로드 성공:", bool(os.getenv("GEMINI_API_KEY")))

## 신규 공고 1건으로 Gemini 호출 테스트

주의: 이 셀은 실제 API를 1회 호출한다. 무료 사용량 안에서 동작하지만, 반복 실행할 때마다 호출이 1회씩 늘어난다.

In [ ]:
# 2) 신규 공고 1건만 Gemini에 보내 한 줄 요약 받기 (8건 한번에 보내지 않음)
from google import genai

GEMINI_MODEL = "gemini-3.5-flash-lite"  # 빠르고 저렴한 안정 버전 모델 (한 줄 요약 테스트용)

if not os.getenv("GEMINI_API_KEY"):
    raise RuntimeError("GEMINI_API_KEY가 없다. .env.example을 복사해서 .env를 만들고 키를 넣은 뒤 위 셀부터 다시 실행한다.")

test_job = df_new.iloc[0]
job_text = (
    f"회사명: {test_job['company_name']}\n"
    f"공고 제목: {test_job['job_title']}\n"
    f"경력 조건: {test_job['career']}"
)
prompt = (
    "다음 채용공고를 한국어 한 문장(50자 이내)으로 요약해줘. "
    "주어진 정보에 없는 내용은 추측해서 덧붙이지 마.\n\n" + job_text
)

client = genai.Client()  # 환경변수 GEMINI_API_KEY를 자동으로 읽는다 (코드에 키를 쓰지 않음)
# 변수명을 response로 하면 STEP 03의 requests 응답(response)을 덮어쓰므로 gemini_response로 둔다
gemini_response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
summary = gemini_response.text.strip()

print("사용 모델:", GEMINI_MODEL)
print("\n[보낸 공고 원문]")
print(job_text)
print("\n[Gemini 한 줄 요약]")
print(summary)

## 실행 결과 해석

> 아래 내용은 노트북 커널에서 위 셀들을 실행한 결과다. API는 1회 호출했다. API 키 값은 출력하지 않았고 여기에도 적지 않는다.

- 성공 여부: 성공. 완료 조건을 만족한다.
- 확인한 데이터/수치:
  - **API 키 로드 성공 여부: True** (`chapter11/.env`에서 로드. 프로젝트 폴더에는 `.env`가 없어서 상위 폴더 파일을 썼다)
  - **사용한 모델명: `gemini-3.5-flash-lite`**
  - **테스트한 공고 1건의 원문** (`df_new` 첫 번째 공고):
    - 회사명: GS리테일
    - 공고 제목: [GS리테일] 9·10월 통합공고(BIZ CLUB팀 경력사원, MD AX 담당, 물류 AX 담당, 보안성 검토 담당)
    - 경력 조건: 경력3년↑
  - **Gemini 응답 요약**: "GS리테일에서 3년 이상 경력자를 대상으로 9·10월 통합공고를 진행합니다."
  - **원문과 대조**: 회사명(GS리테일)과 경력 조건(경력3년↑ → "3년 이상 경력자")이 정확히 반영되었다. 원문에 없는 내용(연봉, 근무지, 자격요건 등)을 추측해서 넣지 않았다.
- 예상과 다른 부분:
  - 틀린 내용은 없다. 다만 요약이 원문 제목의 **모집 직무(BIZ CLUB팀, MD AX 담당, 물류 AX 담당, 보안성 검토 담당)를 전부 빼먹었다.** 이 프로젝트의 관심사인 "AX 직무"라는 핵심 정보가 요약에서 사라진 셈이다. 사실 오류는 아니지만 정보 누락이다.
- 다음 단계 진행 가능 여부: **가능.** 연동(키 로드, SDK, 모델 호출)은 정상 동작한다.
- 추가 확인 사항:
  - STEP 10(Gemini 결과 검증)에서는 "틀린 내용이 없는가"와 함께 "핵심 정보(직무명)를 빠뜨리지 않았는가"도 검증 기준에 넣는다. 필요하면 프롬프트에 "모집 직무를 반드시 포함"을 추가한다.
  - `.env`가 `chapter11/`에 있어 `chapter11/.gitignore`로 커밋을 막아 두었다. 프로젝트 폴더(`.env.example` 옆)로 옮기면 구조가 더 명확해진다.
  - 셀을 다시 실행할 때마다 API 호출이 1회씩 늘어난다.

# STEP 10. Gemini 결과 검증 (원문 대조)

## 작업 계획

STEP 09에서 1건으로 확인한 요약을 신규 공고 8건 전체(`df_new`)로 넓히고, **사람이 원문과 한 건씩 대조**해서 Gemini가 틀리거나 지어낸 내용이 없는지 검증한다.

1. STEP 09의 프롬프트와 모델(`gemini-3.5-flash-lite`), client를 그대로 쓰는 `summarize_job()` 함수를 만든다. 프롬프트는 바꾸지 않는다. STEP 09와 같은 조건에서 8건 결과를 보기 위해서다.
2. `df_new`를 한 건씩 요약한다. 호출 사이에 1초 간격을 둔다. 한 건이 실패해도 나머지는 계속 진행하고, 실패 건은 요약을 비워 둔다.
3. 결과를 `df_summary`(company_name, job_title, career, location, closing_date_parsed, gemini_summary)로 정리한다.
4. 8건을 "원문 → 요약" 순서로 전부 출력한다. 참고용으로 "요약에 회사명이 들어갔는지"만 자동으로 표시하고, **실제 판단은 사람이 원문을 직접 보고 한다.**

검증 기준 (건마다 확인):
- **사실 오류**: 회사명, 경력 조건 등이 원문과 다르게 요약되었는가
- **지어낸 내용(hallucination)**: 원문(회사명·제목·경력)에 없는 정보(연봉, 근무지, 자격요건, 우대사항 등)를 덧붙였는가
- **핵심 정보 누락**: 모집 직무(특히 AX 관련 직무명)를 빠뜨렸는가 (STEP 09에서 발견한 문제)

- 주의: API를 8회 호출한다 (약 8초 이상 걸림). 셀을 다시 실행하면 8회를 다시 호출한다.

완료 조건: 8건 전부 요약 완료, 사람이 원문과 대조한 결과(이상 없음 또는 문제 건수와 내용)를 기록할 것.

In [ ]:
# 1) STEP 09의 프롬프트/모델을 함수로 만들어 신규 공고 전체(df_new)를 한 건씩 요약
import time

SLEEP_SECONDS = 1.0  # API 호출 사이 간격 (과도한 요청 방지)


def build_job_text(job):
    return (
        f"회사명: {job['company_name']}\n"
        f"공고 제목: {job['job_title']}\n"
        f"경력 조건: {job['career']}"
    )


def summarize_job(job):
    # STEP 09와 같은 프롬프트, 같은 모델(GEMINI_MODEL), 같은 client
    prompt = (
        "다음 채용공고를 한국어 한 문장(50자 이내)으로 요약해줘. "
        "주어진 정보에 없는 내용은 추측해서 덧붙이지 마.\n\n" + build_job_text(job)
    )
    response = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    return response.text.strip()


summaries = []
failed = []
for i, (_, job) in enumerate(df_new.iterrows(), start=1):
    try:
        summaries.append(summarize_job(job))
        print(f"[{i}/{len(df_new)}] 완료: {job['company_name']}")
    except Exception as e:  # 한 건이 실패해도 나머지는 계속 진행
        summaries.append(None)
        failed.append((i, job["company_name"], type(e).__name__))
        print(f"[{i}/{len(df_new)}] 실패: {job['company_name']} ({type(e).__name__})")
    if i < len(df_new):
        time.sleep(SLEEP_SECONDS)

df_summary = df_new[["company_name", "job_title", "career", "location", "closing_date_parsed"]].copy()
df_summary["gemini_summary"] = summaries
df_summary = df_summary.reset_index(drop=True)

print("\n사용 모델:", GEMINI_MODEL)
print("요약 성공 건수:", df_summary["gemini_summary"].notna().sum(), "/", len(df_summary))
print("실패 건수:", len(failed), failed)

## 원문 대조용 출력 (8건)

In [ ]:
# 2) 사람이 원문과 대조할 수 있도록 8건을 원문 → 요약 순서로 전부 출력
for i, row in df_summary.iterrows():
    closing = row["closing_date_parsed"]
    closing_text = f"마감 {closing:%Y-%m-%d}" if pd.notna(closing) else "마감일 없음"
    summary = row["gemini_summary"] if pd.notna(row["gemini_summary"]) else "(요약 실패)"

    print(f"[{i + 1}] {row['company_name']} | {row['career']} | {row['location']} | {closing_text}")
    print(f"    원문 제목: {row['job_title']}")
    print(f"    Gemini 요약: {summary}")
    # 사람 검증을 돕는 참고용 표시일 뿐, 판단은 사람이 원문을 직접 보고 한다
    print(f"    (참고) 요약에 회사명 포함: {row['company_name'].lstrip('㈜') in summary}")
    print()

## 실행 결과 해석

> 아래 요약은 노트북 커널에서 위 셀들을 실행한 결과다 (API 8회 호출). 대조 판정은 사람이 원문(회사명·제목·경력)을 직접 보고 한 건씩 확인한 결과다.

- 성공 여부: 성공. 완료 조건을 만족한다.
- 확인한 데이터/수치:
  - 사용 모델: `gemini-3.5-flash-lite` (STEP 09와 같은 프롬프트)
  - **요약 성공 8 / 8, 실패 0**
  - 8건 원문 대비 요약과 사람 대조 결과:

| # | 회사명 | 원문 제목 | 경력 | Gemini 요약 | 사실 오류 | 지어낸 내용 | 직무 누락 |
|---|---|---|---|---|---|---|---|
| 1 | GS리테일 | [GS리테일] 9·10월 통합공고(BIZ CLUB팀 경력사원, MD AX 담당, 물류 AX 담당, 보안성 검토 담당) | 경력3년↑ | GS리테일에서 3년 이상 경력자를 대상으로 BIZ CLUB, MD·물류 AX, 보안성 검토 담당 채용을 진행합니다. | 없음 | 없음 | 없음 |
| 2 | 에스코어 | AX 컨설턴트 채용 | 경력 | 에스코어에서 AX 컨설턴트 경력 사원을 채용합니다. | 없음 | 없음 | 없음 |
| 3 | ㈜NAVER | 광고 프로덕트 기획 경력 채용(AX 광고 예산/과금 구조 설계 담당) | 경력 | ㈜NAVER에서 AX 광고 예산 및 과금 구조 설계를 담당할 경력직 광고 프로덕트 기획자를 채용합니다. | 없음 | 없음 | 없음 |
| 4 | ㈜슈프리마 | [슈프리마HQ] HRD & AX 담당자 모집 | 경력7년↑ | ㈜슈프리마HQ에서 7년 이상 경력의 HRD & AX 담당자를 모집합니다. | 없음 | 없음 | 없음 |
| 5 | ㈜NAVER | 광고 프로덕트 기획 경력 채용(AX 광고 최적화 전략 담당) | 경력 | ㈜NAVER에서 AX 광고 최적화 전략을 담당할 경력직 광고 프로덕트 기획자를 채용합니다. | 없음 | 없음 | 없음 |
| 6 | ㈜캐럿글로벌 | [캐럿글로벌] B2B AI·AX 교육사업 매니저 | 경력3년↑ | 캐럿글로벌에서 3년 이상 경력의 B2B AI·AX 교육사업 매니저를 채용합니다. | 없음 | 없음 | 없음 |
| 7 | ㈜킨다그로스 | [킨다그로스] AX 프로덕트 빌더 | 경력무관 | ㈜킨다그로스에서 경력 무관하게 AX 프로덕트 빌더 채용 공고를 진행 중입니다. | 없음 | 없음 | 없음 |
| 8 | ㈜아시아경제 | [아시아경제] AX전략부 AX전략 및 데이터분석 담당자 채용 | 신입·경력1년↑ | 아시아경제에서 AX전략 및 데이터분석 담당 신입·경력직을 채용합니다. | 없음 | 없음 | 경미 — AX전략부 부서명만 누락 |

  - **집계: 사실 오류 0건, 지어낸 내용(hallucination) 0건, 직무 누락 1건** (8번 ㈜아시아경제. 소속 부서명 "AX전략부"만 빠졌고, 직무 "AX전략 및 데이터분석 담당"은 정확히 반영됨 → 경미)
- 예상과 다른 부분:
  - STEP 09에서 우려했던 "직무명 통째로 누락" 문제가 이번 8건에서는 나타나지 않았다. 같은 GS리테일 공고가 이번에는 직무(BIZ CLUB, MD·물류 AX, 보안성 검토 담당)를 포함해 요약되었다.
- 다음 단계 진행 가능 여부: **가능.** 8건 모두 보고서에 쓸 수 있는 수준이다.
- 추가 확인 사항:
  - **재현성 이슈**: STEP 09 테스트 때는 GS리테일 요약이 직무명을 다 빠뜨렸는데("GS리테일에서 3년 이상 경력자를 대상으로 9·10월 통합공고를 진행합니다."), 이번 STEP 10 실행에서는 잘 포함되었다. 같은 프롬프트, 같은 모델이라도 Gemini 응답은 매번 달라질 수 있다. 프로덕션에서는 이런 변동성을 감안해 **사람이 주기적으로 샘플 검증**하는 것을 권장한다.
  - (참고) 4번 요약의 "㈜슈프리마HQ"는 회사명(㈜슈프리마)과 제목의 "[슈프리마HQ]"를 합친 표현이다. 원문에 있는 정보만 썼으므로 오류로 보지 않았다.
  - 응답이 매번 달라지므로, 보고서(STEP 11)에는 이번에 검증한 `df_summary`의 요약을 그대로 쓴다. 다시 요약하면 검증을 다시 해야 한다.

# STEP 11. Markdown 보고서 생성

## 작업 계획

STEP 10에서 사람이 검증한 `df_summary`(신규 공고 8건 + gemini_summary)로 사람이 읽기 좋은 Markdown 보고서를 만들고 파일로 저장한다. Gemini를 다시 호출하지 않는다. 응답이 매번 달라지므로, 검증을 마친 요약을 그대로 쓴다.

1. 보고서 구성:
   - 제목 "AX 채용정보 주간 리포트", 생성일, 신규 공고 건수, 요약 모델명(`GEMINI_MODEL`)
   - 공고별 항목: 회사명 / 제목 / 경력 / 지역 / 마감일 / Gemini 요약
   - **마감 임박 순** 정렬. 마감일이 없는 공고("상시채용" 등)는 맨 뒤에 두고, 마감일 자리에 원문을 그대로 쓴다. 마감일 옆에 남은 일수(D-n)를 붙인다.
   - 추가: 원문을 바로 확인할 수 있도록 공고 링크(job_url)도 넣는다.
   - `df_summary`에 없는 마감일 원문과 URL은 같은 순서인 `df_new`에서 가져온다. 순서가 어긋나지 않았는지 회사명·제목으로 먼저 확인(assert)한다.
2. `reports/YYYY-MM-DD.md`로 저장한다 (프로젝트 루트 기준). `reports/` 폴더가 없으면 만들고, 같은 날 다시 실행하면 덮어쓴다.
3. 저장한 파일을 다시 읽어서 경로, 크기, 줄 수, 공고 항목 수를 출력한다. 8건의 회사명·제목·요약이 파일에 그대로 들어갔는지 자동으로 세고, 앞부분 내용도 출력한다.

완료 조건: `reports/`에 실제 .md 파일이 생성되고, 그 내용이 8건의 실제 데이터를 정확히 반영할 것.

In [ ]:
# 1) df_summary로 Markdown 보고서 문자열 만들기 (마감 임박 순, 마감일 없는 공고는 맨 뒤)
# df_summary에는 마감일 원문("상시채용" 등)과 공고 URL이 없으므로, 같은 순서인 df_new에서 가져온다
assert (df_summary["company_name"].to_numpy() == df_new["company_name"].to_numpy()).all()
assert (df_summary["job_title"].to_numpy() == df_new["job_title"].to_numpy()).all()

report_df = df_summary.copy()
report_df["closing_date"] = df_new["closing_date"].to_numpy()
report_df["job_url"] = df_new["job_url"].to_numpy()
report_df = report_df.sort_values("closing_date_parsed", na_position="last").reset_index(drop=True)

report_date = pd.Timestamp.today().normalize()


def format_closing(row):
    if pd.isna(row["closing_date_parsed"]):
        return row["closing_date"]  # "상시채용" 등 원문 그대로
    days_left = (row["closing_date_parsed"] - report_date).days
    if days_left > 0:
        d_day = f"D-{days_left}"
    elif days_left == 0:
        d_day = "D-day"
    else:
        d_day = "마감 지남"
    return f"{row['closing_date_parsed']:%Y-%m-%d} ({d_day})"


def build_report_md(report_df):
    lines = [
        "# AX 채용정보 주간 리포트",
        "",
        f"- 생성일: {report_date:%Y-%m-%d}",
        f"- 신규 공고 건수: {len(report_df)}건",
        f"- 요약 모델: {GEMINI_MODEL}",  # STEP 09/10에서 요약에 쓴 모델
        "- 정렬: 마감 임박 순 (마감일 없는 공고는 맨 뒤)",
        "",
        "---",
        "",
    ]
    for i, row in report_df.iterrows():
        lines += [
            f"## {i + 1}. {row['company_name']} — {row['job_title']}",
            "",
            f"- 경력: {row['career']}",
            f"- 지역: {row['location']}",
            f"- 마감일: {format_closing(row)}",
            f"- 요약 (Gemini): {row['gemini_summary']}",
            f"- 공고 링크: {row['job_url']}",
            "",
        ]
    return "\n".join(lines)


report_md = build_report_md(report_df)
print("보고서 줄 수:", len(report_md.splitlines()))
print("보고서 글자 수:", len(report_md))

## 파일로 저장

In [ ]:
# 2) reports/YYYY-MM-DD.md로 저장 (reports 폴더가 없으면 생성, 같은 날 다시 실행하면 덮어씀)
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

report_path = REPORTS_DIR / f"{report_date:%Y-%m-%d}.md"
report_path.write_text(report_md, encoding="utf-8")

print("저장한 파일:", report_path.relative_to(PROJECT_ROOT).as_posix())

## 저장된 보고서 확인

In [ ]:
# 3) 저장된 파일을 다시 읽어서 확인: 경로, 크기, 줄 수, 8건 데이터 반영 여부, 내용 일부
saved_md = report_path.read_text(encoding="utf-8")

print("파일 경로:", report_path.relative_to(PROJECT_ROOT).as_posix())
print("파일 크기:", report_path.stat().st_size, "bytes")
print("줄 수:", len(saved_md.splitlines()))
print("공고 항목 수 ('## ' 제목 줄):", sum(line.startswith("## ") for line in saved_md.splitlines()))

# 8건의 회사명·제목·요약이 파일에 그대로 들어갔는지 확인
for col in ["company_name", "job_title", "gemini_summary"]:
    found = sum(str(value) in saved_md for value in report_df[col])
    print(f"{col} 반영: {found} / {len(report_df)}")

print("\n[보고서 앞부분 (25줄)]")
print("\n".join(saved_md.splitlines()[:25]))

## 실행 결과 해석

> 처음 실행 결과는 71줄 / 3,505 bytes였다. 이후 보고서 상단에 "요약 모델" 줄을 추가하도록 셀 61을 고쳤다. 오늘자 파일은 노트북 밖에서 같은 셀 코드로 다시 생성해 덮어썼다 (STEP 10에서 검증한 요약 그대로, Gemini 재호출 없음). 이전 파일과 비교하면 "요약 모델" 한 줄만 추가되었고 나머지는 완전히 같다. 아래 수치는 다시 생성한 파일 기준이다.

- 성공 여부: 성공. 완료 조건을 만족한다.
- 확인한 데이터/수치:
  - **저장된 파일 경로: `reports/2026-09-23.md`** (프로젝트 루트 기준, `reports/` 폴더 새로 생성)
  - **파일 크기: 3,545 bytes, 72줄** (처음 생성 때 3,505 bytes / 71줄 + 요약 모델 줄)
  - **공고 항목 수: 8** (`## ` 공고 제목 줄 기준)
  - **company_name / job_title / gemini_summary 반영: 각 8 / 8.** 세 항목 모두 8건의 값이 파일에 그대로 들어갔다.
  - **정렬: 마감 임박 순 확인.** 마감일은 아래 순서로 나온다.
    1. ㈜아시아경제 2026-09-28 (D-5)
    2. ㈜NAVER 2026-09-29 (D-6)
    3. ㈜NAVER 2026-09-29 (D-6)
    4. 에스코어 2026-09-30 (D-7)
    5. GS리테일 2026-10-01 (D-8)
    6. ㈜슈프리마 2026-10-25 (D-32)
    7. ㈜킨다그로스 2026-10-31 (D-38)
    8. ㈜캐럿글로벌 "상시채용" (마감일 없음 → 맨 뒤, 원문 그대로 표시)
  - 보고서 내용 일부 (맨 앞 항목):
    ```
    # AX 채용정보 주간 리포트

    - 생성일: 2026-09-23
    - 신규 공고 건수: 8건
    - 요약 모델: gemini-3.5-flash-lite
    - 정렬: 마감 임박 순 (마감일 없는 공고는 맨 뒤)

    ---

    ## 1. ㈜아시아경제 — [아시아경제] AX전략부 AX전략 및 데이터분석 담당자 채용

    - 경력: 신입·경력1년↑
    - 지역: 서울 중구
    - 마감일: 2026-09-28 (D-5)
    - 요약 (Gemini): 아시아경제에서 AX전략 및 데이터분석 담당 신입·경력직을 채용합니다.
    ```
- 예상과 다른 부분:
  - 없음.
- 다음 단계 진행 가능 여부: **가능.** STEP 12(Slack 발송)는 보류하고, STEP 13(Gmail 발송)에서 이 `report_md`(또는 저장된 파일)를 보낸다.
- 추가 확인 사항:
  - 파일명이 실행한 날짜라서, 같은 날 다시 실행하면 같은 파일을 덮어쓴다.
  - D-n 값은 보고서를 만든 날 기준이다. 나중에 파일을 다시 열어 보면 실제 남은 일수와 다를 수 있다.
  - `reports/`는 `.gitignore`에 없어서 주간 보고서도 git에 커밋된다. 원하지 않으면 제외 여부를 정한다.

# STEP 13. Gmail 발송

## 작업 계획

STEP 11에서 만든 보고서(`reports/2026-09-23.md`)를 Gmail로 **본인에게만 1건** 테스트 발송한다. STEP 12(Slack)는 워크스페이스/웹훅 미설정으로 보류했다.

노트북 밖에서 먼저 준비한 것 (완료):
- `.env.example`에 `GMAIL_USER=`, `GMAIL_APP_PASSWORD=` 두 줄 추가 (실제 값 없음). 앱 비밀번호 발급 방법도 주석으로 적었다.
- 실제 값은 `chapter11/.env`에 이미 들어 있다 (변수 이름만 확인, 값은 보지 않음).

이번 셀들에서 할 것:
1. STEP 09의 `.env` 로드 코드(`ENV_CANDIDATES`, `load_dotenv`)를 재사용해 `GMAIL_USER`, `GMAIL_APP_PASSWORD`를 불러온다. **두 값 모두 출력하지 않고, 로드 성공 여부만 True/False로 출력한다.**
2. `smtplib`으로 Gmail SMTP(`smtp.gmail.com`, 587, STARTTLS)에 접속해 메일을 보내는 `send_gmail()` 함수를 만든다. 앱 비밀번호는 로그인에만 쓴다.
3. 보고서 내용을 본문(Markdown 원문을 일반 텍스트로)으로 넣고, 같은 파일을 `.md` 첨부파일로도 붙인다. 제목은 "AX 채용정보 주간 리포트 (2026-09-23)", 받는 사람은 `GMAIL_USER` 본인 1명이다.

절대 하지 않을 것: 비밀번호/앱 비밀번호를 코드에 하드코딩하기, print하기, git에 커밋되는 파일에 남기기. 여러 명에게 보내기.

완료 조건: 발송 성공, 그리고 **본인 메일함에서 실제 수신을 사람이 확인**할 것.

In [ ]:
# 1) STEP 09의 .env 로드 코드를 재사용해 GMAIL_USER, GMAIL_APP_PASSWORD 로드 (값은 절대 출력하지 않는다)
env_path = next((path for path in ENV_CANDIDATES if path.exists()), None)  # ENV_CANDIDATES: STEP 09에서 정의
if env_path is not None:
    load_dotenv(env_path)

print(".env 파일 위치:", env_path.relative_to(PROJECT_ROOT.parent.parent).as_posix() if env_path else "찾지 못함")
print("GMAIL_USER 로드 성공:", bool(os.getenv("GMAIL_USER")))
print("GMAIL_APP_PASSWORD 로드 성공:", bool(os.getenv("GMAIL_APP_PASSWORD")))

## Gmail 발송 함수

In [ ]:
# 2) smtplib + Gmail SMTP(smtp.gmail.com:587, STARTTLS)로 메일 보내는 함수
import smtplib
from email.message import EmailMessage

SMTP_HOST = "smtp.gmail.com"
SMTP_PORT = 587


def send_gmail(to_addr, subject, body, attachment_path=None):
    gmail_user = os.getenv("GMAIL_USER")
    gmail_app_password = os.getenv("GMAIL_APP_PASSWORD")
    if not gmail_user or not gmail_app_password:
        raise RuntimeError("GMAIL_USER / GMAIL_APP_PASSWORD가 없다. .env에 넣은 뒤 위 셀부터 다시 실행한다.")

    msg = EmailMessage()
    msg["From"] = gmail_user
    msg["To"] = to_addr
    msg["Subject"] = subject
    msg.set_content(body)  # 본문: Markdown 원문을 일반 텍스트로
    if attachment_path is not None:
        msg.add_attachment(
            Path(attachment_path).read_bytes(),
            maintype="text",
            subtype="markdown",
            filename=Path(attachment_path).name,
        )

    with smtplib.SMTP(SMTP_HOST, SMTP_PORT, timeout=30) as server:
        server.starttls()
        server.login(gmail_user, gmail_app_password)  # 앱 비밀번호는 여기서만 쓰고 출력하지 않는다
        server.send_message(msg)

## 본인에게 1건 테스트 발송

주의: 이 셀은 실제로 메일을 1통 보낸다. 다시 실행하면 1통이 또 간다.

In [ ]:
# 3) STEP 11 보고서를 본인(GMAIL_USER)에게만 1건 테스트 발송
EMAIL_REPORT_PATH = PROJECT_ROOT / "reports" / "2026-09-23.md"
email_subject = f"AX 채용정보 주간 리포트 ({EMAIL_REPORT_PATH.stem})"
email_body = EMAIL_REPORT_PATH.read_text(encoding="utf-8")

send_gmail(
    to_addr=os.getenv("GMAIL_USER"),  # 받는 사람 = 보내는 사람 본인 (주소는 출력하지 않음)
    subject=email_subject,
    body=email_body,
    attachment_path=EMAIL_REPORT_PATH,  # 같은 보고서를 .md 파일로도 첨부
)

print("발송 성공: True")
print("받는 사람: GMAIL_USER 본인 (1명)")
print("제목:", email_subject)
print("본문:", EMAIL_REPORT_PATH.relative_to(PROJECT_ROOT).as_posix(), f"({len(email_body.splitlines())}줄)", "+ 같은 파일 첨부")

## 실행 결과 해석

> 아래 내용은 노트북 커널에서 위 셀들을 실행하고, 사람이 실제 메일함에서 수신을 확인한 결과다. GMAIL_USER / GMAIL_APP_PASSWORD 값은 적지 않는다.

- 성공 여부: 성공. 완료 조건(발송 성공 + 실제 수신 확인)을 만족한다.
- 확인한 데이터/수치:
  - **GMAIL_USER / GMAIL_APP_PASSWORD 로드 성공: True / True** (값은 미기재)
  - **발송 성공: True** (smtp.gmail.com:587, STARTTLS, 앱 비밀번호 로그인)
  - **받는 사람: k***@gmail.com** (본인, 테스트 발송 1건)
  - **제목: "AX 채용정보 주간 리포트 (2026-09-23)"** — 정상 수신
  - **실제 메일함 확인 결과**:
    - 본문: 보고서 72줄 전체(공고 8건)가 정확히 반영됨
    - 첨부파일: `2026-09-23.md` 포함
    - 한글 깨짐 없음 (본문, 첨부 모두)
    - Gmail 자체 검사 통과
- 예상과 다른 부분:
  - 없음.
- 다음 단계 진행 가능 여부: **가능.** STEP 12(Slack)는 계속 보류 상태이며, 발송 채널은 Gmail로 확보되었다.
- 추가 확인 사항:
  - 본문은 Markdown 원문을 일반 텍스트로 보낸 것이라 `#`, `-` 같은 기호가 그대로 보인다. 읽기 불편하면 HTML 본문으로 바꾸는 것을 나중에 검토한다.
  - 셀 72를 다시 실행하면 메일이 또 1통 간다. 함수화(STEP 14) 이후에는 발송 여부를 명시적으로 켜고 끄는 옵션이 있으면 좋다.
  - 자동화(STEP 17~18) 때는 GMAIL_USER / GMAIL_APP_PASSWORD를 GitHub Secrets에 등록해야 한다.

# STEP 14. 함수화 (src/로 이동)

## 작업 계획

노트북 STEP 03~13에서 검증한 로직을 `src/` 아래 모듈 6개로 옮겼다. 노트북은 프로토타입 기록으로 그대로 둔다. 이번 셀들에서는 **옮긴 코드의 결과가 노트북에서 이미 검증한 값과 정확히 같은지** 비교한다.

| 파일 | 원래 STEP | 주요 함수 |
|---|---|---|
| `src/crawler.py` | 03 / 04 / 04b | `fetch_job_list(keyword, limit=8)` → list[dict] (robots 확인 → 정적 HTML 파싱 → Playwright 날짜 병합) |
| `src/preprocess.py` | 05 / 06 | `clean_jobs(jobs, today=None)` → DataFrame (strip, 연도 처리, job_url 중복 제거) |
| `src/analyzer.py` | 07 / 08 | `detect_new_jobs(df, history_path)`, `update_history(df, history_path)`, `summarize_stats(df_new, today=None, top_n=5)` → dict |
| `src/gemini_client.py` | 09 / 10 | `load_env()`, `summarize_job(job, model=...)` → str, `summarize_jobs(df)` |
| `src/reporter.py` | 11 | `build_report_markdown(df_summary, report_date=None, model=...)` → str, `save_report(markdown, path=None)` |
| `src/notifier.py` | 13 | `send_gmail(subject, body, attachment_path=None, to_addr=None)` |

옮기면서 정리한 것:
- 셀 코드는 그대로 옮겼다. 하드코딩된 기준일은 `today` / `report_date` 파라미터로 바꿨다 (안 주면 오늘 날짜).
- 경로는 노트북 작업 폴더가 아니라 모듈 파일 위치(`Path(__file__)`) 기준으로 잡는다.
- `detect_new_jobs()`(판별만)와 `update_history()`(파일 저장)를 나눴다. 판별만 테스트할 때 실제 history 파일을 건드리지 않기 위해서다.
- `reporter`는 `df_summary`에 closing_date(원문), job_url 컬럼이 있다고 가정한다. 노트북 STEP 11에서는 df_new에서 가져와 붙였던 컬럼들이다.
- API 키, Gmail 앱 비밀번호는 여전히 `.env`에서만 읽는다. import만 해서는 키를 읽거나 API를 호출하지 않는다.

검증 방법 (새 API 호출·메일 발송·수집 요청 없음, 실제 파일은 읽기만 함):
1. 입력: `jobs_history.csv`에 저장된 실제 8건의 원본 7개 컬럼 = STEP 04b 직후 `jobs`와 같은 형태
2. `preprocess` / `analyzer`: 건수, 컬럼, 연도, 신규 판별, 통계를 노트북 STEP 06~08 검증값과 비교한다. history 저장은 임시 폴더에서만 하고, 저장된 내용을 실제 `jobs_history.csv`와 비교한다.
3. `reporter`: STEP 10에서 사람이 검증한 요약 8건으로 보고서를 만들어 `reports/2026-09-23.md`와 **글자 단위로** 비교한다 (파일은 쓰지 않음).
4. `crawler` / `gemini_client` / `notifier`: import, 함수 시그니처, 네트워크 없는 보조 함수만 확인한다 (smoke test). 실제 Gemini 호출과 메일 발송 테스트는 사람이 한다.

완료 조건: src/ 6개 파일 생성, 비교 항목 100% 일치, gemini_client/notifier는 import와 시그니처만 확인.

In [ ]:
# 1) src/ 모듈 불러오기 + 검증 입력 준비 (jobs_history.csv에 저장된 실제 8건, 읽기만 함)
import importlib
import inspect
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.analyzer as analyzer
import src.crawler as crawler
import src.gemini_client as gemini_client
import src.notifier as notifier
import src.preprocess as preprocess
import src.reporter as reporter

for module in [preprocess, analyzer, gemini_client, reporter, notifier, crawler]:
    importlib.reload(module)  # 모듈을 고친 뒤 이 셀만 다시 실행해도 반영되도록

REF_DATE = "2026-09-23"  # 노트북 STEP 06~11을 실행한 기준일
HISTORY_PATH = PROJECT_ROOT / "data" / "processed" / "jobs_history.csv"
history_ref = pd.read_csv(HISTORY_PATH, dtype={"job_id": str}, parse_dates=["posted_date_parsed", "closing_date_parsed"])

RAW_COLUMNS = ["company_name", "job_title", "career", "location", "posted_date", "job_url", "closing_date"]
jobs_input = history_ref[RAW_COLUMNS].to_dict("records")  # STEP 04b 직후의 jobs(list[dict])와 같은 형태

checks = []  # 비교 결과: 모듈, 항목, 기대값, 실제값, 일치


def check(module, item, expected, actual):
    checks.append({"모듈": module, "항목": item, "기대값": expected, "실제값": actual, "일치": expected == actual})


print("src 모듈 import 완료:", [m.__name__ for m in [crawler, preprocess, analyzer, gemini_client, reporter, notifier]])
print("검증 입력 jobs 건수:", len(jobs_input))

## preprocess / analyzer 비교 (STEP 06~08 검증값 기준)

In [ ]:
# 2) preprocess.py / analyzer.py 결과를 노트북에서 검증한 값과 비교 (실제 history 파일은 쓰지 않음)
# --- preprocess.clean_jobs (STEP 06)
df_clean_src = preprocess.clean_jobs(jobs_input, today=REF_DATE)
ref_cols = RAW_COLUMNS + ["posted_date_parsed", "closing_date_parsed"]
check("preprocess", "행 수 (전처리 전 → 후)", "8 → 8", f"{len(jobs_input)} → {len(df_clean_src)}")
check("preprocess", "컬럼", ref_cols, list(df_clean_src.columns))
check("preprocess", "posted_date_parsed 연도", {2026: 8}, df_clean_src["posted_date_parsed"].dt.year.value_counts().to_dict())
check("preprocess", "closing_date_parsed NaT 수", 1, int(df_clean_src["closing_date_parsed"].isna().sum()))
check("preprocess", "마감일 < 등록일 행 수", 0, int((df_clean_src["closing_date_parsed"] < df_clean_src["posted_date_parsed"]).sum()))
try:
    pd.testing.assert_frame_equal(df_clean_src, history_ref[ref_cols], check_dtype=False)
    same_values = True
except AssertionError:
    same_values = False
check("preprocess", "전체 값이 노트북 df_clean(history)과 동일", True, same_values)

# --- analyzer.detect_new_jobs / update_history (STEP 07) - 임시 폴더의 history로만 테스트
with tempfile.TemporaryDirectory() as tmp:
    tmp_history = Path(tmp) / "jobs_history.csv"
    df_first = analyzer.detect_new_jobs(df_clean_src, tmp_history)  # history 없음 = 첫 실행
    saved_rows = analyzer.update_history(df_first, tmp_history)
    tmp_saved = pd.read_csv(tmp_history, dtype={"job_id": str}, parse_dates=["posted_date_parsed", "closing_date_parsed"])
    df_second = analyzer.detect_new_jobs(df_clean_src, tmp_history)  # 같은 데이터로 두 번째 실행

check("analyzer", "job_id 추출 실패(NaN) 수", 0, int(df_first["job_id"].isna().sum()))
check("analyzer", "job_id가 history와 동일", True, df_first["job_id"].tolist() == history_ref["job_id"].tolist())
check("analyzer", "첫 실행 신규 건수", 8, int(df_first["is_new"].sum()))
check("analyzer", "첫 실행 history 저장 행 수", 8, saved_rows)
try:
    pd.testing.assert_frame_equal(tmp_saved, history_ref, check_dtype=False)
    same_history = True
except AssertionError:
    same_history = False
check("analyzer", "저장된 history가 실제 jobs_history.csv와 동일", True, same_history)
check("analyzer", "두 번째 실행 신규 건수", 0, int(df_second["is_new"].sum()))

# --- analyzer.summarize_stats (STEP 08)
df_new_src = df_first[df_first["is_new"]]
stats = analyzer.summarize_stats(df_new_src, today=REF_DATE, top_n=5)
check("analyzer", "신규 공고 건수", 8, stats["new_count"])
check("analyzer", "career 분포", {"경력": 3, "경력3년↑": 2, "경력7년↑": 1, "경력무관": 1, "신입·경력1년↑": 1}, stats["career_counts"])
check("analyzer", "location 분포",
      {"경기 성남시": 3, "서울 강남구 외 1": 1, "서울 송파구": 1, "서울 용산구": 1, "서울 마포구": 1, "서울 중구": 1},
      stats["location_counts"])
check("analyzer", "마감 임박 상위 5건 (회사명)", ["㈜아시아경제", "㈜NAVER", "㈜NAVER", "에스코어", "GS리테일"],
      stats["closing_soon"]["company_name"].tolist())
check("analyzer", "마감 임박 상위 5건 (남은 일수)", [5, 6, 6, 7, 8], stats["closing_soon"]["days_left"].tolist())
check("analyzer", "마감일 없는 공고 수 / 원문", (1, ["상시채용"]), (stats["no_deadline_count"], stats["no_deadline_texts"]))
print("preprocess / analyzer 비교 항목 수:", len(checks))

## reporter 비교 (STEP 11 보고서 파일 기준)

In [ ]:
# 3) reporter.py 결과를 STEP 11 보고서 파일(reports/2026-09-23.md)과 글자 단위로 비교
# STEP 10에서 사람이 원문과 대조해 검증한 요약 8건 (df_new 순서, Gemini 재호출 없음)
VERIFIED_SUMMARIES = [
    "GS리테일에서 3년 이상 경력자를 대상으로 BIZ CLUB, MD·물류 AX, 보안성 검토 담당 채용을 진행합니다.",
    "에스코어에서 AX 컨설턴트 경력 사원을 채용합니다.",
    "㈜NAVER에서 AX 광고 예산 및 과금 구조 설계를 담당할 경력직 광고 프로덕트 기획자를 채용합니다.",
    "㈜슈프리마HQ에서 7년 이상 경력의 HRD & AX 담당자를 모집합니다.",
    "㈜NAVER에서 AX 광고 최적화 전략을 담당할 경력직 광고 프로덕트 기획자를 채용합니다.",
    "캐럿글로벌에서 3년 이상 경력의 B2B AI·AX 교육사업 매니저를 채용합니다.",
    "㈜킨다그로스에서 경력 무관하게 AX 프로덕트 빌더 채용 공고를 진행 중입니다.",
    "아시아경제에서 AX전략 및 데이터분석 담당 신입·경력직을 채용합니다.",
]
df_summary_src = df_new_src.copy()
df_summary_src["gemini_summary"] = VERIFIED_SUMMARIES

report_md_src = reporter.build_report_markdown(df_summary_src, report_date=REF_DATE, model="gemini-3.5-flash-lite")
report_ref = (PROJECT_ROOT / "reports" / f"{REF_DATE}.md").read_text(encoding="utf-8")

check("reporter", "줄 수", len(report_ref.splitlines()), len(report_md_src.splitlines()))
check("reporter", "크기 (bytes)", len(report_ref.encode("utf-8")), len(report_md_src.encode("utf-8")))
check("reporter", "보고서 전체가 STEP 11 파일과 글자 단위로 동일", True, report_md_src == report_ref)
check("reporter", "기본 저장 경로 형식", f"reports/{REF_DATE}.md",
      reporter.default_report_path(REF_DATE).relative_to(PROJECT_ROOT).as_posix())
print("reporter 비교 완료 (파일은 쓰지 않음)")

## crawler / gemini_client / notifier smoke test + 전체 결과

In [ ]:
# 4) crawler / gemini_client / notifier smoke test - import와 함수 시그니처만 확인 (네트워크·API·메일 호출 없음)
def signature_of(func):
    return f"{func.__name__}{inspect.signature(func)}"


check("crawler", "fetch_job_list 시그니처", "fetch_job_list(keyword, limit=8)", signature_of(crawler.fetch_job_list))
check("crawler", "검색 URL 생성", "https://www.jobkorea.co.kr/Search/?stext=AX", crawler.build_search_url("AX"))
sample_card = crawler.BeautifulSoup(
    '<div class="flex flex-shrink-0 gap-[2px]"><span>09/21(월) 등록</span><span>•</span><span>10/01(목) 마감</span></div>',
    "html.parser",
)
check("crawler", "extract_dates (샘플 HTML)", ("09/21(월)", "10/01(목)"), crawler.extract_dates(sample_card))
check("gemini_client", "summarize_job 시그니처", "summarize_job(job, model='gemini-3.5-flash-lite')", signature_of(gemini_client.summarize_job))
check("gemini_client", "기본 모델", "gemini-3.5-flash-lite", gemini_client.GEMINI_MODEL)
check("gemini_client", "프롬프트용 공고 텍스트 (API 호출 없음)",
      "회사명: GS리테일\n공고 제목: [GS리테일] 9·10월 통합공고(BIZ CLUB팀 경력사원, MD AX 담당, 물류 AX 담당, 보안성 검토 담당)\n경력 조건: 경력3년↑",
      gemini_client.build_job_text(df_new_src.iloc[0]))
check("notifier", "send_gmail 시그니처", "send_gmail(subject, body, attachment_path=None, to_addr=None)", signature_of(notifier.send_gmail))
check("notifier", "SMTP 서버", ("smtp.gmail.com", 587), (notifier.SMTP_HOST, notifier.SMTP_PORT))

# 전체 결과
checks_df = pd.DataFrame(checks)
display(checks_df[["모듈", "항목", "일치"]])
print(f"전체 일치: {int(checks_df['일치'].sum())} / {len(checks_df)}")
for _, row in checks_df[~checks_df["일치"]].iterrows():
    print(f"  불일치 [{row['모듈']}] {row['항목']}\n    기대: {row['기대값']}\n    실제: {row['실제값']}")

## 실행 결과 해석

> 아래 수치는 같은 검증 코드를 노트북 밖에서 `notebooks/` 폴더 기준으로 실행한 결과다 (노트북 커널은 쓰지 않음). 실행 중에는 환경변수에서 API 키와 Gmail 값을 뺐고, 실행 전후로 `jobs_history.csv`와 `reports/2026-09-23.md`의 체크섬이 같은 것을 확인했다. 즉 실제 파일은 바뀌지 않았다. 노트북에서 직접 실행하면 같은 결과가 나와야 한다.

- 성공 여부: 성공. 완료 조건을 만족한다.
- 확인한 데이터/수치:
  - **src/ 6개 파일 생성**: crawler.py, preprocess.py, analyzer.py, gemini_client.py, reporter.py, notifier.py
  - **전체 비교 항목 30 / 30 일치 (100%)**

| 모듈 | 확인 항목 | 결과 |
|---|---|---|
| preprocess (6) | 행 수 8 → 8, 컬럼 9개, posted 연도 2026×8, closing NaT 1, 마감<등록 0, **전체 값이 노트북 df_clean과 동일** | 6 / 6 일치 |
| analyzer (12) | job_id 실패 0, job_id가 history와 동일, 첫 실행 신규 8, 저장 8행, **저장 내용이 실제 jobs_history.csv와 동일**, 두 번째 실행 신규 0, 신규 8, career/location 분포, 마감 임박 5건(아시아경제 D-5 → NAVER D-6 ×2 → 에스코어 D-7 → GS리테일 D-8), 마감일 없음 1(상시채용) | 12 / 12 일치 |
| reporter (4) | 72줄, 3,545 bytes, **보고서 전체가 reports/2026-09-23.md와 글자 단위로 동일**, 저장 경로 형식 | 4 / 4 일치 |
| crawler (3) | `fetch_job_list(keyword, limit=8)` 시그니처, 검색 URL 생성, 샘플 HTML 날짜 추출 | 3 / 3 일치 |
| gemini_client (3) | `summarize_job(job, model='gemini-3.5-flash-lite')` 시그니처, 기본 모델, 프롬프트용 공고 텍스트 (API 호출 없음) | 3 / 3 일치 |
| notifier (2) | `send_gmail(subject, body, attachment_path=None, to_addr=None)` 시그니처, smtp.gmail.com:587 | 2 / 2 일치 |

  - 추가 확인 (노트북 셀에는 없음): STEP 04b 때 저장해 둔 실제 렌더링 HTML로 crawler의 파싱 함수를 네트워크 없이 돌렸다. 날짜 8/8이 history와 일치했고, merge 실패 0, 회사명·제목·URL 8/8이 일치했다.
- 예상과 다른 부분:
  - 불일치 없음.
- 다음 단계 진행 가능 여부: **가능.** STEP 15에서 이 모듈들을 `main.py`로 이어 붙인다.
- 추가 확인 사항:
  - **실제 호출 테스트는 사람 몫이다.** `fetch_job_list("AX")`는 요청 2회(정적 1 + Playwright 1)와 robots.txt 1회를 보낸다. `summarize_job()`은 Gemini를 1회 호출하고, `send_gmail()`은 메일을 1통 보낸다. 이번에는 셋 다 호출하지 않았다.
  - **career 셀렉터 변경 (STEP 14 이후 수정)**: `crawler.extract_job()`의 career를 "몇 번째 span" 방식에서 경력 span 전용 셀렉터 `span.flex-shrink-0.text-typo-c1-13`로 바꿨다. 이전 방식은 배지 없는 카드에서 "•"나 렌더링된 날짜를 career로 잡을 위험이 있었다. 바꾼 뒤 비교 결과: 저장된 렌더링 HTML 8/8, 정적 HTML 재현본(렌더링 HTML에서 날짜 span만 제거) 8/8이 STEP 04 정본 career와 일치했다. 위 30개 비교 항목도 다시 30/30 일치했다. STEP 04의 정적 HTML 원본은 파일로 저장된 적이 없어서(커널 메모리의 `response`에만 있었음) 원본으로는 비교하지 못했다.
  - data/raw/search_static.html 저장 시도 중 변수명 충돌로 원본 정적 HTML 유실 확인 → 원인(STEP 09 response 변수 재사용) 수정 완료. 요청 최소화 원칙(PROJECT_SPEC.md 9절)에 따라 재요청하지 않고, 기존 증거(렌더링 HTML 8/8, 정적 HTML 대체본 8/8)로 검증 충분하다고 판단.
  - `notifier`는 `.env` 로드를 위해 `gemini_client.load_env()`를 가져다 쓴다. 설정 모듈이 따로 생기면 그쪽으로 옮긴다.
  - PROJECT_SPEC.md 3절의 `search_keyword`, `collected_at` 컬럼은 여전히 수집 코드에 없다.

## STEP 14-1. src 모듈 실제 호출 테스트 (사람이 실행)

> 📝 중복 검증이라 생략, STEP 14 결과 해석 셀 참고

## 작업 계획

위 비교 검증에서는 네트워크·API·메일을 호출하지 않았다. 이번에는 `src/` 함수 3개를 **실제로 한 번씩** 호출해서 동작을 확인한다.

1. `crawler.fetch_job_list('AX', limit=3)`: 3건만 수집한다. 요청은 robots.txt 1회 + 정적 페이지 1회 + Playwright 1회.
2. `gemini_client.summarize_job()`: `df_new`의 첫 번째 행으로 Gemini를 1회 호출한다. 커널에 `df_new`가 없으면 위 검증 셀의 `df_new_src`(같은 8건)를 쓴다.
3. `notifier.send_gmail()`: 제목에 "[src 테스트]"를 붙여 본인에게 1통 보낸다. 본문은 짧은 안내문이고, `reports/2026-09-23.md`를 첨부한다.

- 전제: 위 STEP 14 셀 1(src 모듈 불러오기)을 먼저 실행해서 `sys.path`에 프로젝트 루트가 들어가 있어야 한다. 테스트 2에서 `df_new_src`를 쓰려면 셀 2까지 실행한다.
- 각 셀을 다시 실행하면 요청 / API 호출 / 메일이 그만큼 또 나간다.
- API 키, Gmail 값은 출력하지 않는다.

완료 조건: 3개 셀이 에러 없이 끝나고, 수집 결과·요약·메일 수신을 사람이 확인할 것.

### 1) 수집 테스트 — `fetch_job_list('AX', limit=3)`

In [ ]:
# 1) crawler.fetch_job_list - 실제 수집 3건 (robots.txt + 정적 요청 1회 + Playwright 1회)
from src.crawler import fetch_job_list

live_jobs = fetch_job_list("AX", limit=3)

print("수집 건수:", len(live_jobs))
for i, job in enumerate(live_jobs, start=1):
    print(f"[{i}] {job['company_name']} | {job['job_title']}")
    print(f"    career={job['career']!r} | location={job['location']!r} | posted_date={job['posted_date']!r} | closing_date={job['closing_date']!r}")
    print(f"    job_url={job['job_url']}")

required = ["company_name", "job_title", "career", "job_url", "posted_date", "closing_date"]
missing = [job for job in live_jobs if not all(job[k] for k in required)]
print("\n값이 비어 있는 공고 수:", len(missing))

### 2) Gemini 요약 테스트 — `summarize_job()` 1회

In [ ]:
# 2) gemini_client.summarize_job - Gemini 1회 호출
from src.gemini_client import GEMINI_MODEL, build_job_text, summarize_job

if "df_new" in globals():
    test_df, source_name = df_new, "df_new"
else:
    test_df, source_name = df_new_src, "df_new_src (STEP 14 검증용, 같은 8건)"
test_job = test_df.iloc[0]

summary_src = summarize_job(test_job)

print("사용 데이터:", source_name)
print("사용 모델:", GEMINI_MODEL)
print("\n[보낸 공고 원문]")
print(build_job_text(test_job))
print("\n[Gemini 한 줄 요약]")
print(summary_src)

### 3) 메일 발송 테스트 — `send_gmail()` 본인에게 1통

In [ ]:
# 3) notifier.send_gmail - 본인에게 1통 발송 (받는 사람을 지정하지 않으면 GMAIL_USER 본인)
from src.notifier import send_gmail

test_report_path = PROJECT_ROOT / "reports" / "2026-09-23.md"
test_subject = f"[src 테스트] AX 채용정보 주간 리포트 ({test_report_path.stem})"
test_body = (
    "src/notifier.py의 send_gmail() 동작 확인용 테스트 메일입니다.\n"
    f"첨부: {test_report_path.name}"
)

send_gmail(subject=test_subject, body=test_body, attachment_path=test_report_path)

print("발송 성공: True")
print("받는 사람: GMAIL_USER 본인 (1명)")
print("제목:", test_subject)
print("첨부:", test_report_path.relative_to(PROJECT_ROOT).as_posix())

## 실행 결과 해석

- 성공 여부: 실행하지 않음 — **중복 검증이라 생략함.** 사유는 STEP 14 결과 해석 셀(82번) 참고.
- 확인한 데이터/수치:
  - fetch_job_list('AX', limit=3): 생략
  - summarize_job: 생략
  - send_gmail: 생략
- 예상과 다른 부분: 없음
- 다음 단계 진행 가능 여부: **가능**
- 추가 확인 사항: 위 3개 셀(85, 87, 89)은 실행하지 않은 상태로 남겨 둔다. 실행하면 실제 요청, Gemini 호출, 메일 발송이 일어난다.

# STEP 12. Slack 발송

## 작업 계획

STEP 12는 워크스페이스/웹훅이 준비되지 않아 보류했다가, STEP 14(함수화) 이후에 진행한다. 그래서 노트북 순서상 맨 뒤에 있다. 발송 함수는 노트북 셀이 아니라 **`src/notifier.py`의 `send_slack(message)`로 바로 만들었다** (STEP 14 구조를 따름).

노트북 밖에서 먼저 준비한 것 (완료):
- `.env.example`에 `SLACK_PROD_WEBHOOK_URL=` 한 줄 추가 (실제 값 없음). 실제 값은 `chapter11/.env`에 이미 있다 (변수 이름만 확인, 값은 보지 않음).
- `src/notifier.py`에 추가한 함수:
  - `send_slack(message)`: `requests.post(webhook_url, json={"text": ...})`로 Incoming Webhook에 1건 보내고 HTTP status code를 돌려준다. 200이 아니면 에러를 낸다.
  - `fit_slack_text(message)`: 3,500자를 넘으면 앞부분만 남기고 "...전체는 이메일 참고"를 붙인다. Slack은 text가 4,000자를 넘으면 자르므로 여유를 뒀다.
  - **Webhook URL 자체가 비밀값**이다. requests의 에러 메시지에는 요청 URL이 들어가므로, 실패하면 URL 없이 "HTTP status + 응답 앞부분"만 담은 에러로 다시 던진다.
- 가짜 Webhook으로 사전 테스트했다 (실제 요청 없음). 성공 시 200 반환. 보고서(2,239자)는 제한 안이라 전체가 그대로 간다. 긴 메시지(6,717자)는 3,500자로 자르고 안내 문구가 붙는다. 404와 연결 오류의 에러 메시지·트레이스백에 URL이 노출되지 않는다. 값이 없으면 발송 전에 멈춘다.

이번 셀들에서 할 것:
1. `.env` 로드 코드(`gemini_client.load_env()`)를 재사용해 `SLACK_PROD_WEBHOOK_URL`을 불러온다. **값은 출력하지 않고, 로드 성공 여부만 True/False로 출력한다.**
2. `reports/2026-09-23.md` 내용을 `send_slack()`으로 Webhook에 연결된 본인 테스트 채널에 **1건만** 보낸다.

- 전제: STEP 14 셀 1(src 모듈 불러오기)을 먼저 실행해서 `sys.path`에 프로젝트 루트가 들어가 있어야 한다.
- 절대 하지 않을 것: Webhook URL을 코드에 하드코딩하기, print하기, git에 커밋되는 파일에 남기기.

완료 조건: HTTP 200, 그리고 **실제 Slack 채널에서 메시지 수신을 사람이 확인**할 것.

In [ ]:
# 1) .env에서 SLACK_PROD_WEBHOOK_URL 로드 (값은 절대 출력하지 않는다)
import os

from src.gemini_client import load_env

slack_env_path = load_env()
print(".env 파일 위치:", slack_env_path.relative_to(PROJECT_ROOT.parent.parent).as_posix() if slack_env_path else "찾지 못함")
print("SLACK_PROD_WEBHOOK_URL 로드 성공:", bool(os.getenv("SLACK_PROD_WEBHOOK_URL")))

## 보고서를 Slack 테스트 채널에 1건 발송

주의: 이 셀은 실제로 Slack에 메시지를 1건 보낸다. 다시 실행하면 1건이 또 간다.

In [ ]:
# 2) STEP 11 보고서를 Slack 테스트 채널에 1건 발송
from src.notifier import SLACK_MAX_CHARS, fit_slack_text, send_slack

slack_report_path = PROJECT_ROOT / "reports" / "2026-09-23.md"
slack_message = slack_report_path.read_text(encoding="utf-8")
slack_text = fit_slack_text(slack_message)  # 실제로 보내질 text (send_slack 안에서도 같은 처리)

slack_status = send_slack(slack_message)

print("발송 HTTP status code:", slack_status)
print("보낸 내용:", slack_report_path.relative_to(PROJECT_ROOT).as_posix())
print(f"글자 수: 보고서 {len(slack_message)}자 → 보낸 text {len(slack_text)}자 (제한 {SLACK_MAX_CHARS}자)")
print("잘림 여부:", slack_text != slack_message)

## 실행 결과 해석

> 아래 내용은 노트북 커널에서 위 셀들을 실행하고, 사람이 실제 Slack 채널에서 수신을 확인한 결과다. SLACK_PROD_WEBHOOK_URL 값은 적지 않는다.

- 성공 여부: 성공. 완료 조건(HTTP 200 + 실제 채널 수신 확인)을 만족한다.
- 확인한 데이터/수치:
  - **SLACK_PROD_WEBHOOK_URL 로드 성공: True** (값은 미기재)
  - **발송 HTTP status code: 200** (`src/notifier.py`의 `send_slack()`)
  - 보낸 내용: `reports/2026-09-23.md` 전체 (2,239자, 3,500자 제한 안이라 잘리지 않음)
  - **실제 채널 수신 확인**: 팀스파르타 AX 2기 워크스페이스 `#팀스파르타-ax-2기-전체` 채널에서 확인. 공고 8건 전체가 반영되었다.
  - 이 채널은 공용 실습 채널(튜터/수강생 12명)이라, 실습 목적으로 사용해도 되는지 확인했다.
- 예상과 다른 부분:
  - Markdown 기호(`#`, `##`, `-`, `---`)는 Slack 특성상 서식으로 바뀌지 않고 일반 텍스트로 그대로 보인다. 내용 전달에는 문제없다.
  - 계획에서는 "본인 테스트용 채널"을 가정했지만, 실제 Webhook은 공용 실습 채널에 연결되어 있다.
- 다음 단계 진행 가능 여부: **가능.**
- 추가 확인 사항:
  - **자동화(STEP 17~18) 때 주의**: 주 1회 자동 실행을 붙이면 이 공용 채널(12명)에 매주 메시지가 올라간다. 테스트나 재실행 때마다 메시지가 가지 않도록, 발송 여부를 켜고 끄는 옵션이나 본인 전용 테스트 Webhook을 따로 두는 것을 검토한다.
  - 가독성이 필요하면 Markdown을 Slack 서식(`*굵게*`, 줄 앞 `•`)으로 바꾸는 변환을 나중에 검토한다.
  - 자동화 때는 `SLACK_PROD_WEBHOOK_URL`을 GitHub Secrets에 등록해야 한다 (PROGRESS.md의 Secrets 목록에는 아직 `SLACK_WEBHOOK_URL`로 적혀 있다).